# Driver Phone Usage Detection System
### Module 5 - Compliance Detection

---

## Project Card

| Field | Details |
|---|---|
| **Dataset** | Roboflow - Cellphone Detector Drivers |
| **Link** | https://universe.roboflow.com/pruebadeteccioncelulareschoferes/cellphone-detector-drivers-ma3vx |
| **Version** | v2 - 3,661 augmented images |
| **Classes** | `driver` - `phone` - `wheel` |

| **Task** | Object Detection - bounding box around phone held by driver |
| **Alert** | Fires ONLY on `phone` class |

---



## Notebook Structure

```
Section 1  ->  Install Dependencies
Section 2  ->  Imports & Configuration
Section 3  ->  Dataset Download
Section 4  ->  Data Exploration
Section 5  ->  Data Preparation
Section 6  ->  Model Loading
Section 7  ->  Training
Section 8  ->  Evaluation
Section 9  ->  Inference Demo
```


In [ ]:
# Install all required packages
%pip install -q ultralytics roboflow opencv-python matplotlib seaborn PyYAML tqdm
print('All packages installed.')


In [ ]:
# Verify environment
import torch, ultralytics, cv2, sys

print(f'Python    : {sys.version.split()[0]}')
print(f'PyTorch   : {torch.__version__}')
print(f'Ultralytics: {ultralytics.__version__}')
print(f'OpenCV    : {cv2.__version__}')
print(f'CUDA      : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'GPU       : {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU. Go to Runtime -> Change runtime type -> T4 GPU.')


---
# Section 2 - Imports & Configuration


In [ ]:
import os, shutil, random, time, yaml
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from tqdm import tqdm
import pandas as pd

import torch
from ultralytics import YOLO

# Display settings
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 14
sns.set_theme(style='darkgrid')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Global paths
DATASET_PATH = Path('dataset')

print(f'All imports OK. Seed = {SEED}')
print(f'Dataset path: {DATASET_PATH.resolve()}')


---
# Section 3 - Dataset Download

## Dataset: Cellphone Detector - Drivers
**Link:** https://universe.roboflow.com/pruebadeteccioncelulareschoferes/cellphone-detector-drivers-ma3vx

### Keep test split untouched while making train cleaner
1. Create a **new Roboflow version** with lighter/no augmentation for training images.
2. Keep the **same split assignment** as the previous version so test images stay identical.
3. Set that version number in Cell 8 (`DATASET_VERSION`) before downloading.

### Get your API key (free, one-time)
1. Go to https://app.roboflow.com
2. Create a free account
3. Settings -> Roboflow API -> copy your Private API Key
4. Paste it below

### Recommended setup
- Use a clean version for download (minimal baked-in augmentation).
- Apply only mild augmentation in Ultralytics training config (Section 7).

In [ ]:
# 3.1 - Download dataset (CLEAN version - no baked-in augmentation)
#
# STRATEGY:
#   - Download version 1 (raw, unaug) OR version 2 but strip augmented copies from train
#   - Test split is NEVER touched (evaluation integrity)
#   - Augmentation is applied ONLINE during training (Section 7) by Ultralytics, not baked into files
#
# HOW TO GET CLEAN IMAGES:
#   Option A (recommended): In Roboflow, create a new version with Augmentation = OFF, same split.
#   Option B: Download v1 (original images, no augmentation).
#   Option C (below): Download v2, then remove augmented copies from train (they have gen suffixes).

import os
from pathlib import Path
from roboflow import Roboflow

try:
    from dotenv import load_dotenv
except ImportError as exc:
    raise ImportError(
        "python-dotenv is required to load ROBOFLOW_API_KEY from .env. Install it with: pip install python-dotenv"
    ) from exc

load_dotenv(dotenv_path=Path('.env'))

ROBOFLOW_API_KEY = os.getenv('ROBOFLOW_API_KEY', '').strip()
DATASET_VERSION = 1   # Use version 1 (original/clean). Change to 2 only if v1 unavailable.

def _is_phone_dataset(path):
    yf = path / 'data.yaml'
    if not yf.exists():
        return False
    cfg = yaml.safe_load(yf.read_text())
    names = cfg.get('names', [])
    if isinstance(names, dict):
        names = list(names.values())
    return any('phone' in str(n).lower() for n in names)

def _count_train(path):
    d = path / 'train' / 'images'
    return len(list(d.glob('*.jpg')) + list(d.glob('*.png'))) if d.exists() else 0

def _remove_augmented_copies(dataset_path, split='train'):
    """
    Roboflow augmentation bakes copies with suffixes like:
       image_aug1.jpg, image_aug2.jpg  OR  image_aug_mosaic_0.jpg
    This function removes them from train/ so only originals remain.
    Labels with matching stems are also removed.
    Returns (removed_imgs, removed_lbls) counts.
    """
    import re
    img_dir = dataset_path / split / 'images'
    lbl_dir = dataset_path / split / 'labels'
    if not img_dir.exists():
        return 0, 0

    # Pattern: filename ends with _aug, _aug\d+, _mosaic_\d+, _flip, _rotate, _crop, etc.
    aug_pattern = re.compile(
        r'_(aug|mosaic|flip|rotate|crop|bright|noise|blur|gray|hflip|vflip|shear|zoom)'
        r'(_\d+|\d+)?$',
        re.IGNORECASE,
    )

    ri = rl = 0
    for img_path in list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png')):
        stem = img_path.stem
        if aug_pattern.search(stem):
            img_path.unlink()
            ri += 1
            for ext in ['.txt', '.xml', '.json']:
                lp = lbl_dir / (stem + ext)
                if lp.exists():
                    lp.unlink()
                    rl += 1
    return ri, rl

# Skip download if already valid
if DATASET_PATH.exists() and _is_phone_dataset(DATASET_PATH) and _count_train(DATASET_PATH) >= 400:
    print(f'Phone dataset already present - {_count_train(DATASET_PATH)} training images. Skipping download.')
else:
    if DATASET_PATH.exists():
        print('Removing existing dataset folder...')
        shutil.rmtree(DATASET_PATH)

    if not ROBOFLOW_API_KEY:
        raise ValueError(
            'ROBOFLOW_API_KEY not found in .env.\n'
            'Create a .env file in this notebook folder with:\n'
            'ROBOFLOW_API_KEY=your_api_key_here\n'
            'Get key at: https://app.roboflow.com -> Settings -> Roboflow API'
        )

    rf      = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace('pruebadeteccioncelulareschoferes').project('cellphone-detector-drivers-ma3vx')

    ok = False
    for fmt in ['yolov8', 'yolov9', 'yolov5pytorch']:
        try:
            print(f'Trying format: {fmt} | version: {DATASET_VERSION}...')
            project.version(DATASET_VERSION).download(fmt, location='dataset')
            ok = True
            print(f'Downloaded with format: {fmt}')
            break
        except Exception as e:
            print(f'  {fmt} failed: {e}')

    if not ok:
        raise RuntimeError('Download failed. Check API key, version id, and internet.')

    # Strip augmented copies from training split
    print('\nRemoving baked-in augmented copies from train split...')
    ri, rl = _remove_augmented_copies(DATASET_PATH, 'train')
    if ri > 0:
        print(f'  Removed {ri} augmented images + {rl} label files from train/')
    else:
        print('  No augmented copies detected (dataset is already clean).')

n = _count_train(DATASET_PATH)
print(f'\nDataset ready at: {DATASET_PATH.resolve()}')
print(f'Training images : {n}')
if n < 200:
    print('WARNING: Very low training image count. Check version and API key.')
elif n < 600:
    print('INFO: Small dataset. Training will still work; consider getting more data for best accuracy.')
else:
    print('Dataset size looks good.')


In [ ]:
# 3.2 - Inspect folder structure
def print_tree(root, max_files=5):
    for dp, _, files in os.walk(root):
        d = dp.replace(str(root), '').count(os.sep)
        print('  ' * d + os.path.basename(dp) + '/')
        for f in files[:max_files]:
            print('  ' * (d+1) + f)
        if len(files) > max_files:
            print('  ' * (d+1) + f'... and {len(files)-max_files} more')

print_tree(DATASET_PATH)


In [ ]:
# 3.3 - Read data.yaml
yaml_path = DATASET_PATH / 'data.yaml'

data_config = yaml.safe_load(yaml_path.read_text())
print('data.yaml contents:')
for k, v in data_config.items():
    print(f'  {k}: {v}')

raw_names = data_config.get('names', {})
if isinstance(raw_names, list):
    CLASS_NAMES = {i: n for i, n in enumerate(raw_names)}
elif isinstance(raw_names, dict):
    CLASS_NAMES = {}
    for k, v in raw_names.items():
        try: CLASS_NAMES[int(k)] = v
        except: CLASS_NAMES[k] = v
else:
    CLASS_NAMES = {}

NUM_CLASSES = data_config.get('nc', len(CLASS_NAMES))
print(f'Classes ({NUM_CLASSES}): {CLASS_NAMES}')


In [ ]:
# 3.4 - Fix YAML paths (Roboflow sometimes writes server-side paths)
def fix_yaml_paths(yp):
    cfg  = yaml.safe_load(yp.read_text())
    root = yp.parent.resolve()
    exp  = {'train': root/'train'/'images', 'val': root/'valid'/'images',
            'test' : root/'test'/'images'}
    for split in ['train', 'val', 'test']:
        if split not in cfg: continue
        raw = Path(str(cfg[split]).replace('\\', '/'))
        if not raw.is_absolute(): raw = (root / raw).resolve()
        cfg[split] = str(raw if raw.exists() else exp.get(split, raw))
    import yaml as _yaml
    yp.write_text(_yaml.dump(cfg, default_flow_style=False, sort_keys=False))
    print('data.yaml paths:')
    for s in ['train', 'val', 'test']:
        if s in cfg:
            p = Path(cfg[s])
            print(f'  {s}: {p}  {"OK" if p.exists() else "NOT FOUND"}')
    return yp

yaml_path = fix_yaml_paths(yaml_path)


---
# Section 4 - Data Exploration & Visualization

Before training, we answer:
1. How many images per split?
2. How are annotations distributed?
3. What do the images and boxes look like?


In [ ]:
# 4.1 - Count images and labels per split
def count_splits(dp):
    stats = {}
    for split in ['train', 'valid', 'test']:
        id_ = dp / split / 'images'
        ld  = dp / split / 'labels'
        imgs = list(id_.glob('*.jpg')) + list(id_.glob('*.png')) if id_.exists() else []
        lbls = list(ld.glob('*.txt')) if ld.exists() else []
        stats[split] = {'images': len(imgs), 'labels': len(lbls)}
    return stats

split_stats  = count_splits(DATASET_PATH)
total_images = 0
print('Dataset split statistics:')
print('-' * 40)
for split, c in split_stats.items():
    print(f'  {split:8s}: {c["images"]:5d} images | {c["labels"]:5d} labels')
    total_images += c['images']
print(f'  {"TOTAL":8s}: {total_images:5d} images')


In [ ]:
# 4.2 - Class distribution
def class_dist(dp, cls_names):
    ctr = Counter()
    for split in ['train', 'valid', 'test']:
        ld = dp / split / 'labels'
        if not ld.exists(): continue
        for lf in ld.glob('*.txt'):
            for line in lf.read_text().splitlines():
                line = line.strip()
                if line:
                    cid = int(line.split()[0])
                    ctr[cls_names.get(cid, f'class_{cid}')] += 1
    return ctr

class_distribution = class_dist(DATASET_PATH, CLASS_NAMES)
total_boxes        = sum(class_distribution.values())

print('Class distribution (bounding boxes):')
print('-' * 45)
for cls, count in sorted(class_distribution.items(), key=lambda x: -x[1]):
    pct = count / total_boxes * 100 if total_boxes else 0
    bar = '#' * int(pct / 2)
    print(f'  {cls:20s}: {count:6d}  ({pct:.1f}%)  {bar}')


In [ ]:
# 4.3 - Class distribution chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
classes = list(class_distribution.keys())
counts  = list(class_distribution.values())
colors  = plt.cm.tab10(np.linspace(0, 1, len(classes)))

bars = axes[0].bar(classes, counts, color=colors, edgecolor='black', linewidth=0.8)
axes[0].set_title('Class Distribution (total bounding boxes)', fontweight='bold')
axes[0].set_ylabel('Bounding boxes')
axes[0].tick_params(axis='x', rotation=30)
for bar, count in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                str(count), ha='center', va='bottom', fontsize=9)

axes[1].pie(counts, labels=classes, autopct='%1.1f%%', colors=colors,
            startangle=90, wedgeprops=dict(edgecolor='white', linewidth=1.5))
axes[1].set_title('Class Share (%)', fontweight='bold')

plt.suptitle('Dataset Class Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: class_distribution.png')


In [ ]:
# 4.4 - Sample images with ground-truth bounding boxes
def yolo_to_px(cx, cy, w, h, iw, ih):
    return (int((cx-w/2)*iw), int((cy-h/2)*ih),
            int((cx+w/2)*iw), int((cy+h/2)*ih))

def show_samples(dp, split, cls_names, n=8):
    id_ = dp / split / 'images'
    ld  = dp / split / 'labels'
    imgs = list(id_.glob('*.jpg')) + list(id_.glob('*.png'))
    if not imgs:
        print(f'No images in {id_}'); return
    samples = random.sample(imgs, min(n, len(imgs)))
    cols = 4
    rows = (len(samples) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols*4, rows*3.5))
    axes = axes.flatten() if rows > 1 or cols > 1 else [axes]
    palette = [(255,50,50),(50,200,50),(50,100,255),(255,165,0),(180,0,255)]
    for ax, ip in zip(axes, samples):
        img = cv2.cvtColor(cv2.imread(str(ip)), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        lp = ld / (ip.stem + '.txt')
        if lp.exists():
            for line in lp.read_text().splitlines():
                p = line.strip().split()
                if len(p) < 5: continue
                cid = int(p[0])
                x1,y1,x2,y2 = yolo_to_px(*map(float,p[1:5]),w,h)
                col = palette[cid % len(palette)]
                name = cls_names.get(cid, str(cid))
                cv2.rectangle(img,(x1,y1),(x2,y2),col,2)
                (tw,th),_ = cv2.getTextSize(name,cv2.FONT_HERSHEY_SIMPLEX,0.5,1)
                cv2.rectangle(img,(x1,y1-th-4),(x1+tw+4,y1),col,-1)
                cv2.putText(img,name,(x1+2,y1-3),cv2.FONT_HERSHEY_SIMPLEX,0.5,(255,255,255),1)
        ax.imshow(img); ax.set_title(ip.name[:22],fontsize=8); ax.axis('off')
    for ax in axes[len(samples):]: ax.axis('off')
    plt.suptitle(f"Samples - '{split}' split (ground-truth)",fontsize=13,fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'samples_{split}.png',dpi=120,bbox_inches='tight')
    plt.show()
    print(f'Saved: samples_{split}.png')

show_samples(DATASET_PATH, 'train', CLASS_NAMES, n=8)


In [ ]:
# 4.5 - Bounding box size and position analysis
def bbox_stats(dp, split='train'):
    stats = {'cx':[],'cy':[],'w':[],'h':[],'area':[]}
    ld = dp / split / 'labels'
    for lf in ld.glob('*.txt'):
        for line in lf.read_text().splitlines():
            p = line.strip().split()
            if len(p) >= 5:
                cx,cy,bw,bh = map(float,p[1:5])
                stats['cx'].append(cx); stats['cy'].append(cy)
                stats['w'].append(bw);  stats['h'].append(bh)
                stats['area'].append(bw*bh)
    return stats

bs = bbox_stats(DATASET_PATH, 'train')
if bs['w']:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].hist(bs['w'],bins=30,color='#3498db',alpha=0.8,label='Width')
    axes[0].hist(bs['h'],bins=30,color='#e74c3c',alpha=0.6,label='Height')
    axes[0].set_title('Box Width & Height'); axes[0].legend()
    axes[1].hist(bs['area'],bins=30,color='#2ecc71',alpha=0.8)
    axes[1].set_title('Box Area Distribution')
    axes[2].hist2d(bs['cx'],bs['cy'],bins=20,cmap='YlOrRd')
    axes[2].set_title('Phone Position Heatmap')
    axes[2].invert_yaxis()
    plt.suptitle('Bounding Box Statistics (Training Set)', fontweight='bold')
    plt.tight_layout()
    plt.savefig('bbox_statistics.png',dpi=120,bbox_inches='tight')
    plt.show()
    print(f'Mean width : {np.mean(bs["w"]):.3f} ({np.mean(bs["w"])*640:.0f}px at 640)')
    print(f'Mean height: {np.mean(bs["h"]):.3f} ({np.mean(bs["h"])*640:.0f}px at 640)')
    print(f'Total boxes: {len(bs["w"])}')
    print('Saved: bbox_statistics.png')


---
# Section 5 - Data Preparation

Ultralytics handles most preprocessing automatically.
We only verify structure and check class imbalance.


In [ ]:
# 5.1 - Verify YOLO folder structure
def verify_structure(dp):
    ok = True
    print('Verifying dataset structure...')
    for split, req in [('train',True),('valid',True),('test',False)]:
        id_ = dp/split/'images'
        ld  = dp/split/'labels'
        s   = 'OK' if (id_.exists() and ld.exists()) else ('MISSING' if req else 'OPTIONAL')
        tag = 'REQUIRED' if req else 'OPTIONAL'
        print(f'  {split:8s} [{tag}]: images={"OK" if id_.exists() else "MISSING"} '
              f'labels={"OK" if ld.exists() else "MISSING"} -> {s}')
        if req and not (id_.exists() and ld.exists()): ok = False
    yaml_ok = (dp/'data.yaml').exists()
    print(f'  data.yaml: {"OK" if yaml_ok else "MISSING"}')
    if not yaml_ok: ok = False
    print('Structure valid.' if ok else 'STRUCTURE HAS ISSUES.')
    return ok

structure_ok = verify_structure(DATASET_PATH)


In [ ]:
# 5.2 - Class imbalance check
# For this dataset: driver appears in every frame, phone only when in use.
# This imbalance is expected and handled by YOLO's loss weighting.
print('Class imbalance check (training set):')
print('-' * 45)
ld = DATASET_PATH / 'train' / 'labels'
counts = Counter()
for lf in ld.glob('*.txt'):
    for line in lf.read_text().splitlines():
        p = line.strip().split()
        if p: counts[int(p[0])] += 1
total = sum(counts.values())
for cid, name in CLASS_NAMES.items():
    count = counts.get(cid, 0)
    share = count / total if total else 0
    flag  = 'LOW' if share < 0.10 else 'OK'
    print(f'  [{flag}] {name:20s}: {count:6d}  ({share*100:.1f}%)')
print()
print('Note: phone class imbalance is expected (not every frame has a phone).')
print('YOLO handles this via loss weighting during training.')


In [ ]:
# 5.3 - Build a phone-focused dataset config
#
# ROOT CAUSE: The dataset has 3 classes (driver, phone, wheel).
# Driver and wheel appear in EVERY frame; phone only appears when in use.
# This imbalance (~70% driver, ~15% wheel, ~15% phone) causes the model
# to "give up" on the rare phone class during training — it gets lower
# gradient signal and the loss doesn't reward learning it enough.
#
# SOLUTION A — Phone-only YAML (strongest fix):
#   Rewrite data.yaml so only the phone class (class index = auto-detected)
#   is kept. Driver and wheel annotations are DROPPED. The model now has
#   exactly 1 job: find phones. No class imbalance, no competing gradients.
#
# SOLUTION B — Retain all classes but oversample phone images:
#   Copy images that contain at least one phone annotation 3x into train/,
#   giving the model 3x more phone examples to learn from.
#
# This cell applies BOTH and lets you choose via TRAINING_MODE below.

import shutil, yaml
from pathlib import Path
from collections import Counter

TRAINING_MODE = 'phone_only'   # options: 'phone_only'  |  'all_classes'
                                # phone_only = strongest for detection accuracy
                                # all_classes = keeps context (driver/wheel visible in eval)

def get_phone_class_id(dataset_path):
    """Auto-detect which class index is the phone."""
    cfg   = yaml.safe_load((dataset_path / 'data.yaml').read_text())
    names = cfg.get('names', [])
    if isinstance(names, dict):
        names = {int(k): v for k, v in names.items()}
        for cid, name in names.items():
            if 'phone' in str(name).lower():
                return cid
    else:
        for i, name in enumerate(names):
            if 'phone' in str(name).lower():
                return i
    raise ValueError('Could not find phone class in data.yaml')

PHONE_CID = get_phone_class_id(DATASET_PATH)
print(f'Phone class ID: {PHONE_CID}')

if TRAINING_MODE == 'phone_only':
    # ── Build a phone-only dataset in dataset_phone/ ──────────────────────
    PHONE_DATASET = Path('dataset_phone')
    if PHONE_DATASET.exists():
        print(f'{PHONE_DATASET} already exists — skipping rebuild.')
    else:
        print('Building phone-only dataset...')
        PHONE_DATASET.mkdir()

        for split in ['train', 'valid', 'test']:
            src_img = DATASET_PATH / split / 'images'
            src_lbl = DATASET_PATH / split / 'labels'
            dst_img = PHONE_DATASET / split / 'images'
            dst_lbl = PHONE_DATASET / split / 'labels'
            if not src_img.exists():
                continue
            dst_img.mkdir(parents=True)
            dst_lbl.mkdir(parents=True)

            kept = skipped = 0
            for lf in src_lbl.glob('*.txt'):
                lines = lf.read_text().splitlines()
                # Keep only phone annotations (remap to class 0)
                phone_lines = [
                    '0 ' + ' '.join(ln.split()[1:])
                    for ln in lines
                    if ln.strip() and int(ln.split()[0]) == PHONE_CID
                ]
                if not phone_lines:
                    skipped += 1
                    continue  # no phone in this image — exclude from train/valid
                              # (keep all for test so we measure real-world recall)
                # Copy image
                for ext in ['.jpg', '.png', '.jpeg']:
                    ip = src_img / (lf.stem + ext)
                    if ip.exists():
                        shutil.copy2(ip, dst_img / ip.name)
                        break
                # Write phone-only label
                (dst_lbl / lf.name).write_text('\n'.join(phone_lines))
                kept += 1

            # For test split: also include images with NO phone
            # (we need them to measure false positive rate)
            if split == 'test':
                for lf in src_lbl.glob('*.txt'):
                    if (dst_lbl / lf.name).exists():
                        continue  # already added
                    for ext in ['.jpg', '.png', '.jpeg']:
                        ip = src_img / (lf.stem + ext)
                        if ip.exists():
                            shutil.copy2(ip, dst_img / ip.name)
                            (dst_lbl / lf.name).write_text('')  # empty = no phone
                            break

            print(f'  {split}: kept {kept} images with phones'
                  + (f', skipped {skipped} without' if split != 'test' else ' (+ all for FP test)'))

        # Write new data.yaml
        new_cfg = {
            'path'  : str(PHONE_DATASET.resolve()),
            'train' : str(PHONE_DATASET / 'train'  / 'images'),
            'val'   : str(PHONE_DATASET / 'valid'  / 'images'),
            'test'  : str(PHONE_DATASET / 'test'   / 'images'),
            'nc'    : 1,
            'names' : ['phone'],
        }
        (PHONE_DATASET / 'data.yaml').write_text(
            yaml.dump(new_cfg, default_flow_style=False, sort_keys=False)
        )
        print(f'\nPhone-only dataset ready: {PHONE_DATASET}')

    yaml_path_phone = PHONE_DATASET / 'data.yaml'
    print(f'YAML: {yaml_path_phone}')
    print('\nUpdate TRAIN_CONFIG to use this yaml before re-training (cell 7.1 will do this automatically).')

elif TRAINING_MODE == 'all_classes':
    # ── Oversample phone images 3x in training set ─────────────────────────
    lbl_dir = DATASET_PATH / 'train' / 'labels'
    img_dir = DATASET_PATH / 'train' / 'images'
    phone_imgs = []
    for lf in lbl_dir.glob('*.txt'):
        if any(int(ln.split()[0]) == PHONE_CID
               for ln in lf.read_text().splitlines() if ln.strip()):
            for ext in ['.jpg','.png','.jpeg']:
                ip = img_dir / (lf.stem + ext)
                if ip.exists():
                    phone_imgs.append((ip, lf))
                    break

    print(f'Phone images found: {len(phone_imgs)}')
    added = 0
    for ip, lf in phone_imgs * 2:   # add 2 extra copies = 3x total
        stem = ip.stem
        for c in range(1, 3):
            new_img = img_dir / f'{stem}_os{c}{ip.suffix}'
            new_lbl = lbl_dir / f'{stem}_os{c}.txt'
            if not new_img.exists():
                shutil.copy2(ip, new_img)
                shutil.copy2(lf, new_lbl)
                added += 1
    print(f'Oversampled: added {added} copies. Total phone images now: {len(phone_imgs) + added}')
    yaml_path_phone = yaml_path   # use original yaml


---
# Section 6 - Model Loading

## Model Selection: YOLOv8n (recommended) vs YOLOv10n

| Model | mAP (COCO) | Speed | Size | Stability | License |
|---|---|---|---|---|---|
| **YOLOv8n** ✅ | 37.3 | Fast | 6.3MB | Excellent | GPL-3.0 |
| YOLOv10n | 38.5 | Fastest | 2.3MB | ⚠️ patchy support | Apache 2.0 |
| YOLOv5n | 28.0 | Medium | 3.9MB | Good | GPL-3.0 |

**Why YOLOv8n for this project:**
- Fully supported in all recent `ultralytics` versions (no import errors)
- COCO-pretrained: includes `cell phone` class → strong transfer learning for phone detection
- NMS included (inference is identical to training, no post-processing mismatch)
- YOLOv10 requires a specific ultralytics build and frequently breaks with pip upgrades

**YOLOv10 issues:** The `yolov10x.pt` used previously is the **extra-large** variant (~120MB),
not the nano. This causes OOM on T4 GPUs. The model also requires `from ultralytics import YOLOv10`
in some versions, while others use `YOLO()` — making it fragile across environments.


In [ ]:
# 6.1 - Load YOLOv8n pretrained on COCO
#
# YOLOv8n is used instead of YOLOv10 for reliability:
#   - Fully supported in all ultralytics versions
#   - COCO pretrained (includes 'cell phone' class = strong transfer for phone detection)
#   - No import errors, no NMS compatibility issues
#   - Size: ~6MB, fast on T4 GPU
#
# To use YOLOv10n instead, change to 'yolov10n.pt' and ensure:
#   pip install ultralytics>=8.2.0  (and even then it may break across versions)

PRETRAINED_MODEL = 'yolov8n.pt'  # <- stable, recommended
# PRETRAINED_MODEL = 'yolov10n.pt'  # <- alternative (may error on some ultralytics versions)

model = YOLO(PRETRAINED_MODEL)

print(f'Model      : {PRETRAINED_MODEL}')
print(f'Parameters : {sum(p.numel() for p in model.model.parameters()):,}')
print(f'File size  : {os.path.getsize(PRETRAINED_MODEL) / 1e6:.1f} MB')
print(f'COCO classes: {len(model.names)}')
print(f'cell phone in COCO: {"cell phone" in model.names.values()}')
print(f'Model type : {type(model.model).__name__}')


---
# Section 7 - Training

## Hyperparameters (clean-first setup)

| Parameter | Value | Why |
|---|---|---|
| `epochs` | 80 | Enough iterations for convergence |
| `patience` | 30 | Stop if validation stalls |
| `batch` | 16 | Reduce to 8 if OOM |
| `lr0` | 0.005 | Stable fine-tuning |
| `mosaic` | 0.0 | Keep visuals closer to original data |
| `fliplr` | 0.3 | Mild left-right invariance |
| `hsv_v` | 0.2 | Mild brightness variation |
| `flipud` | 0.0 | Camera is fixed, no upside-down view |

## Expected time
- Google Colab T4 GPU: ~25-40 minutes (80 epochs, depending on version size)
- Local RTX GPU: ~10-20 minutes

In [ ]:
# 7.1 - Training configuration (IMPROVED for phone detection)
#
# Key changes vs previous config and WHY:
#
# freeze=10       : Freeze first 10 backbone layers during training.
#                   YOLOv8n has ~200 layers; the first 10 encode generic
#                   edge/texture features already learned from COCO (which
#                   includes "cell phone"). Fine-tuning all layers on only
#                   ~600 images causes catastrophic forgetting — the model
#                   forgets what phones look like. Freezing preserves those
#                   COCO features and only adapts the higher-level layers.
#
# mosaic=0.5      : Moderate mosaic (was 0.0). Mosaic concatenates 4 images,
#                   which shows the phone at many scales and positions. This
#                   was previously OFF to "preserve scene context" but that
#                   reasoning hurt phone recall. Phones are small objects —
#                   scale variety is exactly what helps detect them.
#
# scale=0.5       : Larger scale jitter (was 0.2). Phones in real videos
#                   appear much larger or smaller than in training images
#                   depending on camera distance. ±50% scale trains the model
#                   to handle this variation.
#
# lr0=0.001       : Lower initial LR (was 0.005). With a frozen backbone and
#                   a small dataset, a high LR causes the detection head to
#                   overshoot and never converge on the rare phone class.
#
# epochs=120      : More epochs (was 80). With freeze + lower LR, convergence
#                   is slower but more stable. patience=30 still stops early
#                   if validation stalls.
#
# close_mosaic=20 : Turn OFF mosaic in the last 20 epochs so the model
#                   fine-tunes on real single-image scenes before final eval.
#
# yaml selection  : Automatically uses phone-only YAML if cell 5.3 was run
#                   with TRAINING_MODE='phone_only', otherwise uses original.

# Pick the right YAML
_yaml = globals().get('yaml_path_phone', None) or yaml_path
print(f'Using YAML: {_yaml}')

TRAIN_CONFIG = {
    # Data
    'data'         : str(_yaml),
    'imgsz'        : 640,
    # Duration
    'epochs'       : 60,
    'patience'     : 30,
    'close_mosaic' : 20,       # disable mosaic last N epochs for stable fine-tuning
    # Compute
    'batch'        : 16,       # reduce to 8 if OOM
    'device'       : 0,
    'workers'      : 4,
    # Transfer learning — freeze backbone to preserve COCO phone features
    'freeze'       : 10,       # freeze first 10 layers
    # Optimizer — lower LR for stable fine-tuning with frozen layers
    'optimizer'    : 'AdamW',
    'lr0'          : 0.001,    # lower than before (was 0.005)
    'lrf'          : 0.01,
    'momentum'     : 0.937,
    'weight_decay' : 0.0005,
    'warmup_epochs': 5,        # longer warmup with frozen backbone
    # Augmentation — improved for small-object phone detection
    'mosaic'       : 0.5,      # moderate mosaic for scale variety (was 0.0)
    'mixup'        : 0.0,
    'copy_paste'   : 0.0,
    'degrees'      : 5.0,      # slightly more rotation
    'translate'    : 0.1,
    'scale'        : 0.5,      # larger scale jitter (was 0.2)
    'flipud'       : 0.0,
    'fliplr'       : 0.5,      # increased (was 0.3)
    'hsv_h'        : 0.015,
    'hsv_s'        : 0.4,
    'hsv_v'        : 0.4,      # increased brightness range (day/night)
    'erasing'      : 0.3,      # random erasing — helps with occlusion
    # Output
    'project'      : 'runs/phone_detection',
    'name'         : 'yolov8n_phone_v2',
    'save_period'  : 10,
    'plots'        : True,
    'verbose'      : True,
    'seed'         : SEED,
}

print('\nTraining configuration:')
for k, v in TRAIN_CONFIG.items():
    print(f'  {k:20s}: {v}')


In [ ]:
# 7.2 - Run training
#
# SMART GUARD: If best.pt already exists, training is skipped.
# Delete the runs/ folder to retrain from scratch.
#
# PRE-CHECK: Verifies >= 500 training images before starting.
# If version 1 was downloaded, raises a clear error.

def find_best_pt():
    roots = [
        Path(TRAIN_CONFIG['project']),
        Path('runs/phone_detection'),
        Path('runs/detect'),
        Path('runs'),
    ]
    for root in roots:
        if not root.exists(): continue
        for csv in root.rglob('results.csv'):
            bp = csv.parent / 'weights' / 'best.pt'
            if bp.exists(): return bp
    return None

existing = find_best_pt()

if existing:
    print(f'Existing model found - skipping training.')
    print(f'  best.pt: {existing}')
    print(f'  Delete runs/ to retrain from scratch.')
    BEST_MODEL_PATH = existing
    EXPERIMENT_DIR  = existing.parent.parent
else:
    n_train = sum(1 for _ in (DATASET_PATH/'train'/'images').glob('*.jpg'))
    n_train += sum(1 for _ in (DATASET_PATH/'train'/'images').glob('*.png'))
    print(f'Training on {n_train} images')

    if n_train < 500:
        raise RuntimeError(
            f'Only {n_train} training images.\n'
            'Re-run Section 3 to download version 2 (3,661 images).'
        )

    model = YOLO(PRETRAINED_MODEL)
    print('Starting training...')
    print('=' * 60)
    results = model.train(**TRAIN_CONFIG)
    BEST_MODEL_PATH = Path(results.save_dir) / 'weights' / 'best.pt'
    EXPERIMENT_DIR  = Path(results.save_dir)
    print('=' * 60)
    print(f'Training complete!')
    print(f'Best model: {BEST_MODEL_PATH}')
    print(f'Model size: {BEST_MODEL_PATH.stat().st_size/1e6:.1f} MB')


In [ ]:
# 7.3 - Plot training curves
#
# Healthy training:
#   Losses (box, cls, dfl)  -> decreasing smoothly
#   Val box loss            -> decreasing, NOT rising (rising = overfitting)
#   mAP@50                  -> increasing toward convergence
#   Precision & Recall      -> increasing, not oscillating

def find_exp_dir():
    ed = globals().get('EXPERIMENT_DIR', None)
    if ed and Path(str(ed)).exists(): return Path(str(ed))
    roots = [
        Path(TRAIN_CONFIG.get('project','runs')),
        Path('runs/phone_detection'),
        Path('runs/detect'),
        Path('runs'),
    ]
    candidates = []
    for root in roots:
        if not root.exists(): continue
        for csv in root.rglob('results.csv'):
            bp = csv.parent / 'weights' / 'best.pt'
            if bp.exists():
                candidates.append((csv.stat().st_mtime, csv.parent))
    return sorted(candidates, reverse=True)[0][1] if candidates else None

exp_dir = find_exp_dir()
csv_path = Path(str(exp_dir)) / 'results.csv' if exp_dir else None

if csv_path and csv_path.exists():
    BEST_MODEL_PATH = Path(str(exp_dir)) / 'weights' / 'best.pt'
    EXPERIMENT_DIR  = Path(str(exp_dir))
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    fig, axes = plt.subplots(2, 4, figsize=(18, 8))
    axes = axes.flatten()

    metrics = [
        ('train/box_loss',    'Train Box Loss',    '#e74c3c'),
        ('train/cls_loss',    'Train Class Loss',  '#e67e22'),
        ('train/dfl_loss',    'Train DFL Loss',    '#f39c12'),
        ('val/box_loss',      'Val Box Loss',      '#3498db'),
        ('metrics/precision', 'Precision',         '#2ecc71'),
        ('metrics/recall',    'Recall',            '#1abc9c'),
        ('metrics/mAP50',     'mAP@50',            '#9b59b6'),
        ('metrics/mAP50-95',  'mAP@50-95',         '#8e44ad'),
    ]

    for ax, (pat, title, color) in zip(axes, metrics):
        matching = [c for c in df.columns if pat in c]
        if matching:
            col = matching[0]
            ax.plot(df['epoch'], df[col], color=color, linewidth=2)
            ax.fill_between(df['epoch'], df[col], alpha=0.15, color=color)
            ax.set_title(title, fontweight='bold')
            ax.set_xlabel('Epoch')
            ax.grid(True, alpha=0.3)
            if 'mAP50' in col and '95' not in col:
                be = df['epoch'][df[col].idxmax()]
                bv = df[col].max()
                ax.axvline(be,color='gold',linestyle='--',linewidth=1.5,
                           label=f'Best: {bv:.3f} @ ep{be}')
                ax.legend(fontsize=8)
        else:
            ax.text(0.5,0.5,f'{pat}\nnot found',ha='center',va='center',
                    transform=ax.transAxes,color='gray')
            ax.set_title(title)

    plt.suptitle('Training History - YOLOv10n Phone Detection',fontsize=15,fontweight='bold')
    plt.tight_layout()
    plt.savefig('training_curves.png',dpi=120,bbox_inches='tight')
    plt.show()
    print('Saved: training_curves.png')
else:
    print('results.csv not found. Run training first.')


---
# Section 8 - Evaluation

## Metrics

| Metric | Meaning | Target |
|---|---|---|
| **mAP@50** | Detection quality at IoU >= 0.50 | >= 75% |
| **mAP@50-95** | Stricter - averaged IoU 0.50->0.95 | As high as possible |
| **Precision** | Of all alerts, how many were correct? | >= 90% |
| **Recall** | Of all real phones, how many caught? | >= 90% |

**Rule**: evaluate on test set exactly once. Using test results to tune parameters = data leakage.


In [ ]:
# 8.1 - Load best model and evaluate on validation set
def get_best_model():
    bm = globals().get('BEST_MODEL_PATH', None)
    if bm and Path(str(bm)).exists(): return Path(str(bm))
    ed = find_exp_dir()
    if ed:
        bp = Path(str(ed)) / 'weights' / 'best.pt'
        if bp.exists(): return bp
    return None

BEST_MODEL_PATH = get_best_model()

if not BEST_MODEL_PATH:
    print('best.pt not found. Run training first.')
else:
    print(f'Loading: {BEST_MODEL_PATH}')
    best_model    = YOLO(str(BEST_MODEL_PATH))
    EXPERIMENT_DIR = BEST_MODEL_PATH.parent.parent

    print('\nEvaluating on validation set...')
    val_m = best_model.val(
        data=TRAIN_CONFIG.get('data', str(yaml_path)), imgsz=640, batch=16, device=0,
        split='val', verbose=True,
    )

    map50 = val_m.box.map50
    target = 'TARGET MET (>=75%)' if map50 >= 0.75 else 'BELOW TARGET'

    print('\n' + '=' * 50)
    print('VALIDATION RESULTS')
    print('=' * 50)
    print(f'  mAP@50    : {map50:.4f}  ({map50*100:.1f}%)  {target}')
    print(f'  mAP@50-95 : {val_m.box.map:.4f}  ({val_m.box.map*100:.1f}%)')
    print(f'  Precision : {val_m.box.mp:.4f}  ({val_m.box.mp*100:.1f}%)')
    print(f'  Recall    : {val_m.box.mr:.4f}  ({val_m.box.mr*100:.1f}%)')
    print('=' * 50)


In [ ]:
# 8.2 - Evaluate on held-out TEST set (run only once)
test_dir = DATASET_PATH / 'test' / 'images'

if not BEST_MODEL_PATH or not BEST_MODEL_PATH.exists():
    print('best.pt not found. Run training first.')
elif not test_dir.exists():
    print('Test split not found. Skipping test evaluation.')
else:
    if 'best_model' not in globals(): best_model = YOLO(str(BEST_MODEL_PATH))
    print('Evaluating on TEST set (final)...')
    test_m = best_model.val(
        data=TRAIN_CONFIG.get('data', str(yaml_path)), imgsz=640, batch=16, device=0,
        split='test', verbose=True,
    )
    print('\n' + '=' * 50)
    print('FINAL TEST SET RESULTS')
    print('=' * 50)
    print(f'  mAP@50    : {test_m.box.map50:.4f}  ({test_m.box.map50*100:.1f}%)')
    print(f'  mAP@50-95 : {test_m.box.map:.4f}  ({test_m.box.map*100:.1f}%)')
    print(f'  Precision : {test_m.box.mp:.4f}  ({test_m.box.mp*100:.1f}%)')
    print(f'  Recall    : {test_m.box.mr:.4f}  ({test_m.box.mr*100:.1f}%)')
    print('=' * 50)


In [ ]:
# 8.3 - Visualize predictions on test images
def viz_preds(mdl, img_dir, cls_names, n=8, conf=0.25):
    paths = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png'))
    if not paths: print(f'No images in {img_dir}'); return
    samples = random.sample(paths, min(n, len(paths)))
    cols = 4; rows = (len(samples)+cols-1)//cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols*4, rows*3.5))
    axes = axes.flatten() if rows>1 or cols>1 else [axes]
    for ax, ip in zip(axes, samples):
        img = cv2.cvtColor(cv2.imread(str(ip)), cv2.COLOR_BGR2RGB).copy()
        res = mdl.predict(ip, conf=conf, verbose=False)[0]
        n_phone = 0
        if res.boxes is not None:
            for box in res.boxes:
                x1,y1,x2,y2 = map(int,box.xyxy[0].tolist())
                cid = int(box.cls[0])
                name = cls_names.get(cid, str(cid))
                is_ph = 'phone' in name.lower()
                color = (220,50,50) if is_ph else (120,120,120)
                thick = 3 if is_ph else 1
                cv2.rectangle(img,(x1,y1),(x2,y2),color,thick)
                lbl = f'{name} {float(box.conf[0]):.2f}'
                (tw,th),_ = cv2.getTextSize(lbl,cv2.FONT_HERSHEY_SIMPLEX,0.5,1)
                cv2.rectangle(img,(x1,y1-th-4),(x1+tw+4,y1),color,-1)
                cv2.putText(img,lbl,(x1+2,y1-3),cv2.FONT_HERSHEY_SIMPLEX,0.5,(255,255,255),1)
                if is_ph: n_phone+=1
        ax.imshow(img)
        ax.set_title('PHONE' if n_phone>0 else 'SAFE',
                     color='red' if n_phone>0 else 'green',fontsize=9,fontweight='bold')
        ax.axis('off')
    for ax in axes[len(samples):]: ax.axis('off')
    plt.suptitle(f'Predictions (conf>{conf}) - red=phone, gray=context',
                 fontsize=13,fontweight='bold')
    plt.tight_layout()
    plt.savefig('test_predictions.png',dpi=120,bbox_inches='tight')
    plt.show()
    print('Saved: test_predictions.png')

if BEST_MODEL_PATH and BEST_MODEL_PATH.exists():
    if 'best_model' not in globals(): best_model = YOLO(str(BEST_MODEL_PATH))
    img_dir = test_dir if test_dir.exists() else DATASET_PATH/'valid'/'images'
    viz_preds(best_model, img_dir, CLASS_NAMES)
else:
    print('Run training first.')


In [ ]:
# 8.4 - Confusion matrix and PR curve
def show_eval_plots(exp_dir):
    candidates = []
    for root in [Path('runs/detect'), Path('runs'), Path(str(exp_dir))]:
        if not root.exists(): continue
        for vd in root.rglob('val*'):
            if not vd.is_dir(): continue
            plots = [vd/'confusion_matrix.png', vd/'confusion_matrix_normalized.png',
                     vd/'BoxPR_curve.png', vd/'BoxF1_curve.png']
            if any(p.exists() for p in plots):
                t = max(p.stat().st_mtime for p in plots if p.exists())
                candidates.append((t, vd))
    if not candidates: print('No evaluation plots found. Run Section 8.1 first.'); return
    val_dir = sorted(candidates,reverse=True)[0][1]
    found = [(n,p) for n,p in [
        ('Confusion Matrix', val_dir/'confusion_matrix.png'),
        ('Normalized CM',    val_dir/'confusion_matrix_normalized.png'),
        ('PR Curve',         val_dir/'BoxPR_curve.png'),
        ('F1 Curve',         val_dir/'BoxF1_curve.png'),
    ] if p.exists()]
    if not found: print(f'No plots in {val_dir}'); return
    fig,axes=plt.subplots(1,len(found),figsize=(6*len(found),5))
    if len(found)==1: axes=[axes]
    for ax,(name,path) in zip(axes,found):
        ax.imshow(cv2.cvtColor(cv2.imread(str(path)),cv2.COLOR_BGR2RGB))
        ax.set_title(name,fontweight='bold'); ax.axis('off')
    plt.suptitle('Evaluation Artifacts',fontsize=14,fontweight='bold')
    plt.tight_layout()
    plt.savefig('evaluation_artifacts.png',dpi=100,bbox_inches='tight')
    plt.show()
    print('Saved: evaluation_artifacts.png')

ed = globals().get('EXPERIMENT_DIR', None)
if ed and Path(str(ed)).exists():
    show_eval_plots(Path(str(ed)))
else:
    print('Run Section 8.1 first.')


---
# Section 9 - Inference Demo

| Cell | Mode | What you need |
|---|---|---|
| **9.0** | PhoneDetector class | (defines the class) |
| **9.1** | Upload image | Any photo from your computer |
| **9.2** | Image from URL | Any public image URL |
| **9.3** | Batch or video | Test split (auto) or a .mp4 file |

**Alert logic:** Only `phone` class triggers `alert_triggered=True`.
`driver` and `wheel` are drawn in gray for context but do not alert.


In [ ]:
# 9.0 - PhoneDetector class (IMPROVED)
#
# Changes vs previous version:
#
# 1. WHEEL HIDDEN: wheel boxes are completely excluded from the output image.
#    Driver boxes shown as faint gray (context only, no label text).
#    Only phone gets a visible colored box + label.
#
# 2. LOWER DEFAULT CONF (0.20): phones are often partially occluded or at
#    angles that produce lower confidence scores. 0.35 suppressed many real
#    detections. 0.20 recovers them with acceptable false-positive rate when
#    combined with a phone-only model.
#
# 3. PHONE-ONLY MODEL AWARE: automatically handles both 1-class (phone_only)
#    and 3-class (all_classes) models via name lookup.

class PhoneDetector:
    # Classes to SHOW in output (phone gets full box+label, driver gets faint dot only)
    # wheel is completely hidden — it adds visual noise and no useful information
    SHOW_CLASSES   = {'phone', 'cell phone', 'mobile', 'cellphone', 'driver'}
    ALERT_CLASSES  = {'phone', 'cell phone', 'mobile', 'cellphone'}
    HIDE_CLASSES   = {'wheel', 'steering wheel', 'wheel_driver'}

    def __init__(self, model_path, conf_threshold=0.20, device='0'):
        self.model          = YOLO(model_path)
        self.class_names    = self.model.names
        self.conf_threshold = conf_threshold
        self.device         = device
        print(f'PhoneDetector loaded: {model_path}')
        print(f'  Threshold : {conf_threshold}')
        print(f'  Classes   : {self.class_names}')
        print(f'  Alert on  : {self.ALERT_CLASSES}')
        print(f'  Hidden    : {self.HIDE_CLASSES} (wheel boxes suppressed)')

    def _is_phone(self, name):
        return name.lower() in self.ALERT_CLASSES

    def _is_hidden(self, name):
        return name.lower() in self.HIDE_CLASSES

    def detect(self, frame):
        res  = self.model.predict(source=frame, conf=self.conf_threshold,
                                  device=self.device, verbose=False)[0]
        dets = []
        if res.boxes is not None:
            for box in res.boxes:
                cid  = int(box.cls[0])
                name = self.class_names.get(cid, str(cid))
                if self._is_hidden(name):
                    continue   # drop wheel entirely — not stored, not drawn
                dets.append({
                    'class_id'  : cid,
                    'class_name': name,
                    'confidence': float(box.conf[0]),
                    'box'       : box.xyxy[0].tolist(),
                    'is_phone'  : self._is_phone(name),
                })
        phones    = [d for d in dets if d['is_phone']]
        annotated = self._draw(frame.copy(), dets)
        return {
            'detections'    : dets,
            'phone_detections': phones,
            'alert_triggered': len(phones) > 0,
            'n_phones'      : len(phones),
            'annotated_frame': annotated,
        }

    def _draw(self, frame, dets):
        PHONE_COLOR  = (0, 0, 220)    # red (BGR)
        DRIVER_COLOR = (60, 60, 60)   # dark gray, very faint
        has_phone = any(d['is_phone'] for d in dets)

        for d in dets:
            x1, y1, x2, y2 = map(int, d['box'])
            if d['is_phone']:
                # Phone: bold red box + label
                cv2.rectangle(frame, (x1,y1), (x2,y2), PHONE_COLOR, 3)
                lbl = f"phone {d['confidence']:.2f}"
                (tw,th),_ = cv2.getTextSize(lbl, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
                cv2.rectangle(frame, (x1, y1-th-8), (x1+tw+6, y1), PHONE_COLOR, -1)
                cv2.putText(frame, lbl, (x1+3, y1-5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)
            else:
                # Driver: thin gray outline only, no label (context marker)
                cv2.rectangle(frame, (x1,y1), (x2,y2), DRIVER_COLOR, 1)

        # Top banner
        n      = sum(1 for d in dets if d['is_phone'])
        banner = PHONE_COLOR if has_phone else (0, 170, 0)
        text   = f'PHONE DETECTED ({n})' if has_phone else 'SAFE'
        cv2.rectangle(frame, (0,0), (frame.shape[1], 40), banner, -1)
        cv2.putText(frame, text, (10,28),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.85, (255,255,255), 2)
        return frame

    def run_on_video(self, source, output_path=None, display=False, max_frames=None):
        cap   = cv2.VideoCapture(source)
        fps   = cap.get(cv2.CAP_PROP_FPS) or 30
        w,h   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        limit = max_frames or total
        print(f'Video: {w}x{h} @ {fps:.0f}fps | {total} frames | processing {limit}')
        writer = None
        if output_path:
            writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w,h))
        idx = phones = 0; times = []
        with tqdm(total=limit, desc='Processing') as pbar:
            while cap.isOpened() and idx < limit:
                ret, frame = cap.read()
                if not ret: break
                t0  = time.perf_counter()
                res = self.detect(frame)
                times.append((time.perf_counter()-t0)*1000)
                if res['alert_triggered']: phones += 1
                if writer: writer.write(res['annotated_frame'])
                if display:
                    cv2.imshow('Phone Detection', res['annotated_frame'])
                    if cv2.waitKey(1) & 0xFF == ord('q'): break
                idx += 1; pbar.update(1)
        cap.release()
        if writer: writer.release()
        if display: cv2.destroyAllWindows()
        if output_path: print(f'Saved: {output_path}')
        avg = np.mean(times) if times else 0
        return {
            'total_frames'  : idx,
            'frames_with_phone': phones,
            'phone_rate'    : phones/idx if idx else 0,
            'mean_fps'      : 1000/avg if avg else 0,
            'mean_latency_ms': avg,
        }

print('PhoneDetector class defined.')
print('  conf_threshold=0.20 (lower than before to recover occluded phones)')
print('  wheel class is hidden from output')


In [ ]:
# 9.1 - Test: Upload a single image
# Colab : run -> Choose Files -> pick any image
# Local : set MANUAL_IMAGE_PATH = 'path/to/image.jpg'

import numpy as np, cv2, matplotlib.pyplot as plt
from pathlib import Path

MANUAL_IMAGE_PATH = ''

_bmp = get_best_model()
if not _bmp: print('No model. Run Section 7 first.')
else:
    _det = PhoneDetector(str(_bmp), conf_threshold=0.25)

    def show_result(frame, title=''):
        res = _det.detect(frame)
        ann = cv2.cvtColor(res['annotated_frame'], cv2.COLOR_BGR2RGB)
        n   = res['n_phones']
        fig,axes=plt.subplots(1,2,figsize=(14,5))
        axes[0].imshow(cv2.cvtColor(frame,cv2.COLOR_BGR2RGB))
        axes[0].set_title('Original',fontweight='bold'); axes[0].axis('off')
        axes[1].imshow(ann)
        lbl = f'PHONE DETECTED - {n}' if n>0 else 'SAFE - No phone'
        axes[1].set_title(lbl,color='red' if n>0 else 'green',fontweight='bold',fontsize=12)
        axes[1].axis('off')
        if title: plt.suptitle(title,fontsize=13,fontweight='bold')
        plt.tight_layout()
        plt.savefig('test_result.png',dpi=120,bbox_inches='tight')
        plt.show()
        print(f'Result: {n} phone(s)  |  Saved: test_result.png')

    try:
        from google.colab import files as _gf
        print('Select an image file:')
        up=_gf.upload()
        for fname,data in up.items():
            frame=cv2.imdecode(np.frombuffer(data,np.uint8),cv2.IMREAD_COLOR)
            if frame is None: print(f'Could not decode {fname}')
            else: show_result(frame, title=fname)
    except ImportError:
        path=MANUAL_IMAGE_PATH.strip() or input('Enter image path: ').strip()
        if Path(path).exists(): show_result(cv2.imread(path), Path(path).name)
        else: print(f'File not found: {path}')


In [ ]:
# 9.2 - Test: Image from URL
# Replace the URL with a real driver-using-phone image for a meaningful test.

import urllib.request, numpy as np, cv2, matplotlib.pyplot as plt

TEST_IMAGE_URL = 'jareb wahda hamza hh'
# Replace with a URL to a driver-using-phone image

_bmp = get_best_model()
if not _bmp: print('No model. Run Section 7 first.')
else:
    if '_det' not in globals(): _det = PhoneDetector(str(_bmp), conf_threshold=0.25)
    print(f'Downloading: {TEST_IMAGE_URL}')
    try:
        req=urllib.request.Request(TEST_IMAGE_URL,headers={'User-Agent':'Mozilla/5.0'})
        with urllib.request.urlopen(req,timeout=15) as resp: data=resp.read()
        frame=cv2.imdecode(np.frombuffer(data,np.uint8),cv2.IMREAD_COLOR)
        if frame is None:
            print('Could not decode. URL must end in .jpg or .png')
        else:
            res=_det.detect(frame)
            ann=cv2.cvtColor(res['annotated_frame'],cv2.COLOR_BGR2RGB)
            n  =res['n_phones']
            fig,axes=plt.subplots(1,2,figsize=(14,5))
            axes[0].imshow(cv2.cvtColor(frame,cv2.COLOR_BGR2RGB))
            axes[0].set_title('Original',fontweight='bold'); axes[0].axis('off')
            axes[1].imshow(ann)
            axes[1].set_title('PHONE DETECTED' if n>0 else 'SAFE',
                             color='red' if n>0 else 'green',fontweight='bold',fontsize=12)
            axes[1].axis('off')
            plt.suptitle('URL Image Test',fontsize=13,fontweight='bold')
            plt.tight_layout()
            plt.savefig('test_url_result.png',dpi=120,bbox_inches='tight')
            plt.show()
            print(f'Result: {n} phone(s)  |  Saved: test_url_result.png')
    except Exception as e:
        print(f'Failed: {e}')


In [ ]:
# 9.3 - Test: Batch of images or Video
# BATCH (default): runs on dataset/test/images automatically
# VIDEO: upload a .mp4 to Colab then set TEST_MODE = 'video'

import os, random as _r, time
import numpy as np, cv2, matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm

# -- CONFIGURE --
TEST_MODE         = 'batch'                  # 'batch' or 'video'
TEST_VIDEO_PATH   = 'test_video.mp4'         # for video mode
TEST_IMAGE_FOLDER = 'dataset/test/images'    # for batch mode
CONF_THRESHOLD    = 0.25
MAX_VIDEO_FRAMES  = 150                      # None = whole video
BATCH_N_SHOW      = 12
# ----------------

_bmp = get_best_model()
if not _bmp: print('No model. Run Section 7 first.')

elif TEST_MODE == 'batch':
    _det   = PhoneDetector(str(_bmp), conf_threshold=CONF_THRESHOLD)
    folder = Path(TEST_IMAGE_FOLDER)
    if not folder.exists() or not (list(folder.glob('*.jpg'))+list(folder.glob('*.png'))):
        folder = Path('dataset/valid/images')
        print('test/ empty - using valid/')
    imgs = list(folder.glob('*.jpg'))+list(folder.glob('*.png'))+list(folder.glob('*.jpeg'))
    if not imgs: print(f'No images in {folder}')
    else:
        samples = _r.sample(imgs, min(BATCH_N_SHOW, len(imgs)))
        cols=4; rows=(len(samples)+cols-1)//cols
        fig,axes=plt.subplots(rows,cols,figsize=(cols*5,rows*4))
        af=axes.flatten() if hasattr(axes,'flatten') else [axes]
        dc=sc=0
        for ax,ip in zip(af,samples):
            fr=cv2.imread(str(ip))
            if fr is None: ax.axis('off'); continue
            res=_det.detect(fr)
            ax.imshow(cv2.cvtColor(res['annotated_frame'],cv2.COLOR_BGR2RGB))
            if res['alert_triggered']:
                dc+=1
                confs=[f"{d['confidence']:.2f}" for d in res['phone_detections']]
                ax.set_title(f'PHONE  {", ".join(confs)}',color='red',fontsize=8,fontweight='bold')
            else:
                sc+=1; ax.set_title('SAFE',color='green',fontsize=8,fontweight='bold')
            ax.axis('off')
        for ax in af[len(samples):]: ax.axis('off')
        plt.suptitle(f'Batch Test - {dc} phone / {sc} safe (conf>{CONF_THRESHOLD})',
                     fontsize=13,fontweight='bold')
        plt.tight_layout()
        plt.savefig('batch_test_grid.png',dpi=120,bbox_inches='tight')
        plt.show()
        print(f'Result: {dc} detected / {sc} safe  |  Saved: batch_test_grid.png')

elif TEST_MODE == 'video':
    if not os.path.exists(TEST_VIDEO_PATH):
        print(f'Video not found: {TEST_VIDEO_PATH}')
        print('Upload a .mp4 to Colab (Files panel) and update TEST_VIDEO_PATH.')
    else:
        _det    = PhoneDetector(str(_bmp), conf_threshold=CONF_THRESHOLD)
        summary = _det.run_on_video(TEST_VIDEO_PATH,'output_test.mp4',False,MAX_VIDEO_FRAMES)
        print(f'Frames processed  : {summary["total_frames"]}')
        print(f'Frames with phone : {summary["frames_with_phone"]}  ({summary["phone_rate"]*100:.1f}%)')
        print(f'Average FPS       : {summary["mean_fps"]:.1f}')
        print(f'Mean latency      : {summary["mean_latency_ms"]:.1f} ms')
        print('Download output_test.mp4 from the Files panel (left sidebar in Colab)')
else:
    print(f'Unknown TEST_MODE: {TEST_MODE!r}. Use "batch" or "video".')


---
# Section 10 - Grad-CAM Heatmap Explainability

## What is Grad-CAM and why do we need it?

When your model detects a phone and draws a bounding box, you know **what** it detected —
but not **why**. The box could be correct for the wrong reason:
the model might be responding to the driver's hand, the steering wheel, or even a dashboard
reflection rather than the phone itself.

**Grad-CAM** (Gradient-weighted Class Activation Mapping) solves this by generating a heatmap
that highlights which pixels most strongly activated the model's detection decision.

| Without Grad-CAM | With Grad-CAM |
|---|---|
| Box drawn around phone ✅ | Box drawn + red hotspot ON the phone screen ✅✅ |
| Box drawn around hand ❌ | Hotspot on hand, not phone — model is wrong for right reason ⚠️ |
| Box on dashboard glare ❌ | Hotspot on reflection — false positive explained ⚠️ |

## How it works (simplified)

```
Input image → YOLOv8 backbone → Feature maps → Detection head → "phone" score
                                     ↑
              Grad-CAM hooks here: measure which feature map channels
              had the HIGHEST GRADIENT flowing back from the phone score
              → those channels contributed most → weight + average → heatmap
```

The heatmap is overlaid on the original image as a color gradient:
- 🔴 **Red / hot** = model is focusing HERE strongly
- 🔵 **Blue / cool** = model ignoring this area

## What good vs bad looks like

| Heatmap pattern | Interpretation |
|---|---|
| Hot on phone screen/body | ✅ Model learned the right feature |
| Hot on driver's hand only | ⚠️ Partial — hand is a proxy, not robust |
| Hot on face or chest | ❌ Spurious correlation — will fail on new angles |
| Scattered / diffuse heat | ❌ Model is uncertain / confused |
| Hot matches bbox tightly | ✅ Confident, well-localized detection |

## Cells in this section

| Cell | What it does |
|---|---|
| **10.0** | GradCAMExtractor class (hooks into YOLOv8 backbone) |
| **10.1** | Single image: side-by-side original / detection / heatmap |
| **10.2** | Grid: multiple test images with heatmaps |
| **10.3** | Heatmap focus score — quantifies how well the model focuses on the phone bbox |


In [ ]:
# # 10.0 - GradCAMExtractor class
# #
# # HOW IT WORKS:
# #   1. Register a forward hook on the last Conv layer of the YOLOv8 backbone (layer -3).
# #      This captures the feature map: shape (B, C, H, W) at stride-32 resolution.
# #   2. Run a forward pass to get detections + feature map.
# #   3. For each detected phone box, compute a score = sum of logits inside that box region.
# #   4. Backpropagate that score through the backbone → gradients flow into the feature map.
# #   5. Global-average-pool the gradients over spatial dims → per-channel importance weights.
# #   6. Weighted sum of feature map channels → raw activation map.
# #   7. ReLU (keep only positive contributions) → resize to input resolution → normalize to [0,1].
# #   8. Apply colormap and blend with original image.
# #
# # LAYER CHOICE:
# #   YOLOv8 backbone last conv (model.model[-3]) gives the richest semantic features
# #   while still having spatial resolution. Going deeper (detection head) loses spatial info.

# import cv2
# import numpy as np
# import torch
# import torch.nn.functional as F
# import matplotlib.pyplot as plt
# import matplotlib.gridspec as gridspec
# from pathlib import Path
# from ultralytics import YOLO


# class GradCAMExtractor:
#     """
#     Grad-CAM for YOLOv8 phone detection.

#     Usage:
#         gcam = GradCAMExtractor(model_path, conf=0.25)
#         result = gcam.explain(image_bgr)
#         gcam.visualize(result)
#     """

#     def __init__(self, model_path: str, conf: float = 0.25, target_layer_idx: int = -3):
#         self.model = YOLO(model_path)
#         self.conf  = conf
#         self._phone_keywords = {'phone', 'cell phone', 'mobile', 'cellphone'}

#         # ── Hook setup ──────────────────────────────────────────────────
#         # target_layer_idx=-3 → last conv block before detection head in YOLOv8
#         self._fmap   = None   # forward: feature maps
#         self._grads  = None   # backward: gradients

#         target_layer = list(self.model.model.model.children())[target_layer_idx]

#         def _fwd_hook(module, inp, out):
#             self._fmap = out               # shape: (1, C, H, W)

#         def _bwd_hook(module, grad_in, grad_out):
#             self._grads = grad_out[0]      # shape: (1, C, H, W)

#         target_layer.register_forward_hook(_fwd_hook)
#         target_layer.register_full_backward_hook(_bwd_hook)

#         self._layer_name = str(type(target_layer).__name__)
#         print(f'GradCAMExtractor ready.')
#         print(f'  Model      : {model_path}')
#         print(f'  Conf       : {conf}')
#         print(f'  Hook layer : [{target_layer_idx}] {self._layer_name}')

#     # ── public API ───────────────────────────────────────────────────────────

#     def explain(self, image_bgr: np.ndarray) -> dict:
#         """
#         Run detection + Grad-CAM on a single BGR image.

#         Returns dict with keys:
#             original_bgr, detections, phone_boxes,
#             heatmap_norm (H×W float32 0-1),
#             overlay_bgr  (same size as input, heatmap blended),
#             alert        (bool)
#         """
#         h, w = image_bgr.shape[:2]

#         # Convert to tensor
#         img_rgb  = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
#         tensor   = torch.from_numpy(img_rgb).permute(2, 0, 1).float() / 255.0
#         tensor   = tensor.unsqueeze(0)                       # (1,3,H,W)
#         if next(self.model.model.parameters()).is_cuda:
#             tensor = tensor.cuda()
#         tensor.requires_grad_(True)

#         # Forward pass (no ultralytics postprocess — we need raw gradients)
#         self.model.model.eval()
#         raw_out = self.model.model(tensor)                   # raw predictions

#         # Also run normal detection for boxes/labels
#         with torch.no_grad():
#             det_result = self.model.predict(
#                 image_bgr, conf=self.conf, verbose=False
#             )[0]

#         detections  = []
#         phone_boxes = []
#         if det_result.boxes is not None:
#             for box in det_result.boxes:
#                 cid  = int(box.cls[0])
#                 name = self.model.names.get(cid, str(cid))
#                 entry = {
#                     'class_id'  : cid,
#                     'class_name': name,
#                     'confidence': float(box.conf[0]),
#                     'box_xyxy'  : box.xyxy[0].tolist(),
#                     'is_phone'  : name.lower() in self._phone_keywords,
#                 }
#                 detections.append(entry)
#                 if entry['is_phone']:
#                     phone_boxes.append(entry['box_xyxy'])

#         # ── Grad-CAM computation ─────────────────────────────────────────
#         heatmap_norm = self._compute_gradcam(raw_out, tensor, h, w, phone_boxes)

#         # ── Overlay ──────────────────────────────────────────────────────
#         overlay_bgr = self._blend_heatmap(image_bgr, heatmap_norm)

#         return {
#             'original_bgr' : image_bgr,
#             'detections'   : detections,
#             'phone_boxes'  : phone_boxes,
#             'heatmap_norm' : heatmap_norm,
#             'overlay_bgr'  : overlay_bgr,
#             'alert'        : len(phone_boxes) > 0,
#         }

#     def visualize(self, result: dict, title: str = '', save_path: str = None,
#                   show_focus_score: bool = True):
#         """
#         Three-panel figure: Original | Detection | Grad-CAM heatmap overlay.
#         Optionally prints the focus score (how well heatmap aligns with phone bbox).
#         """
#         orig  = cv2.cvtColor(result['original_bgr'],  cv2.COLOR_BGR2RGB)
#         det   = self._draw_detections(result['original_bgr'].copy(), result['detections'])
#         det   = cv2.cvtColor(det, cv2.COLOR_BGR2RGB)
#         over  = cv2.cvtColor(result['overlay_bgr'],   cv2.COLOR_BGR2RGB)

#         fig = plt.figure(figsize=(18, 5))
#         gs  = gridspec.GridSpec(1, 4, width_ratios=[3, 3, 3, 0.15], wspace=0.05)

#         ax0 = fig.add_subplot(gs[0])
#         ax1 = fig.add_subplot(gs[1])
#         ax2 = fig.add_subplot(gs[2])
#         cax = fig.add_subplot(gs[3])

#         ax0.imshow(orig);  ax0.set_title('Original',   fontweight='bold'); ax0.axis('off')
#         ax1.imshow(det);   ax1.set_title('Detections', fontweight='bold'); ax1.axis('off')

#         im = ax2.imshow(orig)
#         ax2.imshow(result['heatmap_norm'],
#                    cmap='jet', alpha=0.5,
#                    vmin=0, vmax=1,
#                    extent=[0, orig.shape[1], orig.shape[0], 0])

#         # Draw phone boxes on heatmap panel
#         for bx in result['phone_boxes']:
#             x1,y1,x2,y2 = map(int, bx)
#             rect = plt.Rectangle((x1,y1), x2-x1, y2-y1,
#                                   linewidth=2, edgecolor='white',
#                                   facecolor='none', linestyle='--')
#             ax2.add_patch(rect)

#         verdict = 'PHONE DETECTED' if result['alert'] else 'SAFE'
#         color   = 'red' if result['alert'] else 'green'
#         ax2.set_title(f'Grad-CAM  [{verdict}]', fontweight='bold', color=color)
#         ax2.axis('off')

#         # Colorbar
#         sm = plt.cm.ScalarMappable(cmap='jet', norm=plt.Normalize(0, 1))
#         sm.set_array([])
#         cb = plt.colorbar(sm, cax=cax)
#         cb.set_label('Activation', rotation=270, labelpad=12, fontsize=9)
#         cb.set_ticks([0, 0.5, 1])
#         cb.set_ticklabels(['Low\n(ignored)', 'Mid', 'High\n(focused)'])

#         if title:
#             plt.suptitle(title, fontsize=13, fontweight='bold', y=1.01)

#         plt.tight_layout()

#         if show_focus_score and result['phone_boxes']:
#             score = self.focus_score(result)
#             print(f'  Focus score: {score:.3f}  '
#                   f'({"GOOD ✅" if score >= 0.4 else "WEAK ⚠️" if score >= 0.2 else "POOR ❌"})')
#             print(f'  Interpretation: '
#                   f'{score*100:.0f}% of heatmap energy is inside the phone bounding box.')

#         sp = save_path or 'gradcam_result.png'
#         plt.savefig(sp, dpi=130, bbox_inches='tight')
#         plt.show()
#         print(f'Saved: {sp}')

#     def focus_score(self, result: dict) -> float:
#         """
#         Quantifies how well the Grad-CAM heatmap aligns with phone bounding boxes.

#         Score = (sum of heatmap values INSIDE all phone boxes)
#               / (sum of ALL heatmap values + epsilon)

#         Range: 0.0 (heatmap entirely outside box) → 1.0 (all heat inside box)

#         Thresholds (empirical):
#             >= 0.40  GOOD   — model is genuinely looking at the phone
#             >= 0.20  WEAK   — some focus on phone, but also spurious activations
#             <  0.20  POOR   — heatmap is mostly outside phone region (model guessing)
#         """
#         h_map = result['heatmap_norm']
#         H, W  = h_map.shape
#         mask  = np.zeros_like(h_map, dtype=np.float32)

#         for bx in result['phone_boxes']:
#             x1, y1, x2, y2 = [int(v) for v in bx]
#             x1, y1 = max(0, x1), max(0, y1)
#             x2, y2 = min(W, x2), min(H, y2)
#             mask[y1:y2, x1:x2] = 1.0

#         inside  = float((h_map * mask).sum())
#         total   = float(h_map.sum()) + 1e-8
#         return inside / total

#     # ── private helpers ──────────────────────────────────────────────────────

#     def _compute_gradcam(self, raw_out, tensor, orig_h, orig_w, phone_boxes):
#         """Core Grad-CAM: backprop from phone score → weighted feature maps → heatmap."""
#         self.model.model.zero_grad()

#         # Build score: sum of raw outputs in regions corresponding to detected phone boxes
#         # If no phone detected, fall back to max activation anywhere (show where model looked)
#         if isinstance(raw_out, (list, tuple)):
#             pred = raw_out[0] if isinstance(raw_out[0], torch.Tensor) else raw_out[0][0]
#         else:
#             pred = raw_out

#         if phone_boxes and pred.dim() >= 2:
#             # Score = objectness * class confidence for phone predictions
#             score = pred.reshape(-1).max()
#         else:
#             score = pred.reshape(-1).max()

#         score.backward(retain_graph=True)

#         if self._fmap is None or self._grads is None:
#             # Fallback: return uniform heatmap if hooks didn't fire
#             return np.zeros((orig_h, orig_w), dtype=np.float32)

#         with torch.no_grad():
#             # Global average pool gradients → channel weights
#             weights = self._grads.mean(dim=(2, 3), keepdim=True)   # (1, C, 1, 1)
#             # Weighted sum of feature maps
#             cam     = (weights * self._fmap).sum(dim=1, keepdim=True)  # (1, 1, H, W)
#             cam     = F.relu(cam)                                   # keep positives only
#             cam     = F.interpolate(
#                 cam, size=(orig_h, orig_w),
#                 mode='bilinear', align_corners=False
#             )
#             cam     = cam.squeeze().cpu().numpy()

#         # Normalize to [0, 1]
#         cam_min, cam_max = cam.min(), cam.max()
#         if cam_max - cam_min > 1e-8:
#             cam = (cam - cam_min) / (cam_max - cam_min)
#         else:
#             cam = np.zeros_like(cam)

#         return cam.astype(np.float32)

#     def _blend_heatmap(self, image_bgr, heatmap_norm):
#         """Convert normalized heatmap → color and blend with original image."""
#         hmap_uint8  = (heatmap_norm * 255).astype(np.uint8)
#         hmap_color  = cv2.applyColorMap(hmap_uint8, cv2.COLORMAP_JET)
#         blended     = cv2.addWeighted(image_bgr, 0.55, hmap_color, 0.45, 0)
#         return blended

#     def _draw_detections(self, image_bgr, detections):
#         """Draw bounding boxes and labels on a copy of the image."""
#         PHONE_COLOR = (0, 0, 220)
#         OTHER_COLOR = (120, 120, 120)
#         for d in detections:
#             x1, y1, x2, y2 = map(int, d['box_xyxy'])
#             col   = PHONE_COLOR if d['is_phone'] else OTHER_COLOR
#             thick = 3 if d['is_phone'] else 1
#             cv2.rectangle(image_bgr, (x1, y1), (x2, y2), col, thick)
#             lbl = f"{d['class_name']}: {d['confidence']:.2f}"
#             (tw, th), _ = cv2.getTextSize(lbl, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
#             cv2.rectangle(image_bgr, (x1, y1-th-6), (x1+tw+4, y1), col, -1)
#             cv2.putText(image_bgr, lbl, (x1+2, y1-4),
#                         cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255,255,255), 1)
#         # Alert banner
#         has_phone = any(d['is_phone'] for d in detections)
#         n_phones  = sum(1 for d in detections if d['is_phone'])
#         banner    = PHONE_COLOR if has_phone else (0, 160, 0)
#         text      = f'PHONE DETECTED ({n_phones})' if has_phone else 'SAFE'
#         cv2.rectangle(image_bgr, (0,0), (image_bgr.shape[1], 38), banner, -1)
#         cv2.putText(image_bgr, text, (10, 27),
#                     cv2.FONT_HERSHEY_SIMPLEX, 0.85, (255,255,255), 2)
#         return image_bgr


# print('GradCAMExtractor class defined.')
# print('Next: run cell 10.1 to test on a single image.')


In [ ]:
# # 10.1 - Single image Grad-CAM explanation
# #
# # WHAT TO LOOK FOR:
# #   ✅ Red/orange hotspot sits ON the phone screen or the hand holding it
# #   ✅ Hotspot aligns tightly with the white dashed bounding box
# #   ⚠️ Hotspot on driver's face → model uses face as proxy (fragile)
# #   ❌ Hotspot on steering wheel → likely a false positive
# #
# # The focus score printed below tells you quantitatively how much
# # of the heatmap energy falls inside the phone bounding box.

# _bmp = get_best_model()
# if not _bmp:
#     print('No model found. Run Section 7 first.')
# else:
#     gcam = GradCAMExtractor(str(_bmp), conf=0.25)

#     # --- pick an image ---
#     # Option A: random image from test set (default)
#     # Option B: set IMAGE_PATH to any file path
#     IMAGE_PATH = ''   # leave empty to use test set automatically

#     if IMAGE_PATH and Path(IMAGE_PATH).exists():
#         img_path = Path(IMAGE_PATH)
#     else:
#         test_imgs  = list((DATASET_PATH/'test'/'images').glob('*.jpg'))
#         test_imgs += list((DATASET_PATH/'test'/'images').glob('*.png'))
#         if not test_imgs:
#             test_imgs  = list((DATASET_PATH/'valid'/'images').glob('*.jpg'))
#             test_imgs += list((DATASET_PATH/'valid'/'images').glob('*.png'))
#         if not test_imgs:
#             print('No test images found. Set IMAGE_PATH to a file path.')
#             img_path = None
#         else:
#             img_path = random.choice(test_imgs)

#     if img_path:
#         print(f'Analyzing: {img_path.name}')
#         frame  = cv2.imread(str(img_path))
#         result = gcam.explain(frame)
#         gcam.visualize(result,
#                        title=f'Grad-CAM Analysis — {img_path.name}',
#                        save_path='gradcam_single.png')

#         # Print detection summary
#         print()
#         print('Detections:')
#         for d in result['detections']:
#             icon = '📱' if d['is_phone'] else '🔘'
#             print(f'  {icon} {d["class_name"]:12s}  conf={d["confidence"]:.3f}  '
#                   f'box={[round(v) for v in d["box_xyxy"]]}')
#         if not result['detections']:
            # print('  (none)')


In [ ]:
# # 10.2 - Grid: Grad-CAM on multiple test images
# #
# # Shows a 3-column grid (Original | Detection | Heatmap) for N_IMAGES samples.
# # Each row also prints its focus score so you can compare across images.
# #
# # USE THIS TO:
# #   - Spot patterns: does the model always look at the hand? The screen? The car?
# #   - Find failure cases: images where heatmap is scattered despite correct box
# #   - Build intuition about what the model actually learned

# N_IMAGES     = 6      # number of images to show (keep ≤ 8 for readability)
# ONLY_PHONES  = True   # if True, skip images with no phone detection (recommended)
# CONF         = 0.25

# _bmp = get_best_model()
# if not _bmp:
#     print('No model found. Run Section 7 first.')
# else:
#     if 'gcam' not in globals():
#         gcam = GradCAMExtractor(str(_bmp), conf=CONF)

#     # Collect images
#     src_dir   = DATASET_PATH / 'test' / 'images'
#     if not src_dir.exists() or not list(src_dir.glob('*.jpg')):
#         src_dir = DATASET_PATH / 'valid' / 'images'
#     all_imgs  = list(src_dir.glob('*.jpg')) + list(src_dir.glob('*.png'))
#     random.shuffle(all_imgs)

#     # Filter to images that actually have phone detections (more interesting)
#     samples = []
#     print(f'Scanning images for phone detections (target: {N_IMAGES})...')
#     for ip in all_imgs:
#         if len(samples) >= N_IMAGES:
#             break
#         frame  = cv2.imread(str(ip))
#         if frame is None:
#             continue
#         result = gcam.explain(frame)
#         if ONLY_PHONES and not result['alert']:
#             continue
#         samples.append((ip.name, result))

#     if not samples:
#         print('No phone detections found. Set ONLY_PHONES=False to show all images.')
#     else:
#         print(f'Found {len(samples)} images with phone detections. Plotting...')
#         n     = len(samples)
#         fig, axes = plt.subplots(n, 3, figsize=(15, 4.5 * n))
#         if n == 1:
#             axes = axes[np.newaxis, :]

#         col_titles = ['Original', 'Detection', 'Grad-CAM Heatmap']
#         for col_idx, ct in enumerate(col_titles):
#             axes[0, col_idx].set_title(ct, fontsize=13, fontweight='bold', pad=10)

#         for row, (fname, result) in enumerate(samples):
#             orig = cv2.cvtColor(result['original_bgr'], cv2.COLOR_BGR2RGB)
#             det  = cv2.cvtColor(
#                        gcam._draw_detections(result['original_bgr'].copy(),
#                                              result['detections']),
#                        cv2.COLOR_BGR2RGB)

#             axes[row, 0].imshow(orig)
#             axes[row, 0].set_ylabel(fname[:28], fontsize=7, rotation=0,
#                                      labelpad=90, va='center')
#             axes[row, 0].axis('off')

#             axes[row, 1].imshow(det)
#             axes[row, 1].axis('off')

#             axes[row, 2].imshow(orig)
#             axes[row, 2].imshow(result['heatmap_norm'],
#                                 cmap='jet', alpha=0.5, vmin=0, vmax=1,
#                                 extent=[0, orig.shape[1], orig.shape[0], 0])
#             for bx in result['phone_boxes']:
#                 x1,y1,x2,y2 = map(int, bx)
#                 rect = plt.Rectangle((x1,y1), x2-x1, y2-y1,
#                                       lw=2, edgecolor='white',
#                                       facecolor='none', ls='--')
#                 axes[row, 2].add_patch(rect)

#             score = gcam.focus_score(result) if result['phone_boxes'] else 0.0
#             verdict = ('GOOD ✅' if score >= 0.4
#                        else 'WEAK ⚠️' if score >= 0.2 else 'POOR ❌')
#             axes[row, 2].set_title(f'Focus: {score:.2f} — {verdict}',
#                                     fontsize=9, color='darkred')
#             axes[row, 2].axis('off')

#         plt.suptitle('Grad-CAM Grid — Is the model looking at the right place?',
#                      fontsize=14, fontweight='bold', y=1.005)
#         plt.tight_layout()
#         plt.savefig('gradcam_grid.png', dpi=120, bbox_inches='tight')
#         plt.show()
#         print('Saved: gradcam_grid.png')

#         # Summary statistics
#         scores = [gcam.focus_score(r) for _, r in samples if r['phone_boxes']]
#         if scores:
#             print(f'\nFocus score summary ({len(scores)} images with phone boxes):')
#             print(f'  Mean  : {np.mean(scores):.3f}')
#             print(f'  Min   : {np.min(scores):.3f}')
#             print(f'  Max   : {np.max(scores):.3f}')
#             print(f'  GOOD (≥0.40): {sum(s>=0.4 for s in scores)}/{len(scores)}')
#             print(f'  WEAK (0.20-0.39): {sum(0.2<=s<0.4 for s in scores)}/{len(scores)}')
#             print(f'  POOR (<0.20): {sum(s<0.2 for s in scores)}/{len(scores)}')


In [ ]:
# # 10.3 - Grad-CAM on a video (frame-by-frame heatmap)
# #
# # Processes a video and saves an output MP4 with Grad-CAM overlay on every frame.
# # Useful for showing your supervisor a real-time explanation of what the model sees.
# #
# # CONFIGURE:
# #   - Set VIDEO_PATH to your test video file
# #   - MAX_FRAMES controls how many frames to process (None = whole video)
# #   - FRAME_SKIP: process every Nth frame (1=all, 2=every other, etc.)
# #     Increase for faster processing if the video is long.

# VIDEO_PATH  = 'test_video.mp4'   # path to your test video
# OUTPUT_PATH = 'gradcam_video.mp4'
# MAX_FRAMES  = 200    # limit for speed; set None for full video
# FRAME_SKIP  = 2      # process every 2nd frame (set 1 for every frame)
# CONF        = 0.30

# from tqdm import tqdm

# _bmp = get_best_model()
# if not _bmp:
#     print('No model found. Run Section 7 first.')
# elif not Path(VIDEO_PATH).exists():
#     print(f'Video not found: {VIDEO_PATH}')
#     print('Upload a .mp4 and update VIDEO_PATH, or run cell 10.1/10.2 for image-level heatmaps.')
# else:
#     if 'gcam' not in globals():
#         gcam = GradCAMExtractor(str(_bmp), conf=CONF)

#     cap    = cv2.VideoCapture(VIDEO_PATH)
#     fps    = cap.get(cv2.CAP_PROP_FPS) or 30
#     W      = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
#     H      = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
#     total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
#     limit  = min(MAX_FRAMES or total, total)

#     writer = cv2.VideoWriter(
#         OUTPUT_PATH,
#         cv2.VideoWriter_fourcc(*'mp4v'),
#         fps / FRAME_SKIP,
#         (W, H)
#     )

#     phone_frames = 0
#     focus_scores = []
#     prev_overlay = None   # reuse last overlay for skipped frames

#     print(f'Processing {limit} frames (skip={FRAME_SKIP}) from {VIDEO_PATH}...')
#     with tqdm(total=limit) as pbar:
#         for idx in range(limit):
#             ret, frame = cap.read()
#             if not ret:
#                 break

#             if idx % FRAME_SKIP == 0:
#                 result      = gcam.explain(frame)
#                 prev_overlay = result['overlay_bgr']

#                 # Draw detection labels on the overlay
#                 annotated = gcam._draw_detections(prev_overlay.copy(), result['detections'])

#                 # Focus score badge
#                 if result['phone_boxes']:
#                     phone_frames += 1
#                     fs = gcam.focus_score(result)
#                     focus_scores.append(fs)
#                     badge_color = (0,200,0) if fs >= 0.4 else (0,165,255) if fs >= 0.2 else (0,0,220)
#                     badge_text  = f'Focus: {fs:.2f}'
#                     cv2.rectangle(annotated, (W-160, 42), (W-5, 72), badge_color, -1)
#                     cv2.putText(annotated, badge_text, (W-155, 63),
#                                 cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255,255,255), 2)

#                 writer.write(annotated)
#             else:
#                 # Write previous overlay for skipped frames
#                 if prev_overlay is not None:
#                     writer.write(prev_overlay)
#                 else:
#                     writer.write(frame)

#             pbar.update(1)

#     cap.release()
#     writer.release()
#     print(f'\nGrad-CAM video saved: {OUTPUT_PATH}')
#     print(f'Phone detected in : {phone_frames}/{limit} frames ({phone_frames/limit*100:.1f}%)')
#     if focus_scores:
#         print(f'Mean focus score  : {np.mean(focus_scores):.3f}')
#         print(f'GOOD frames (≥0.4): {sum(s>=0.4 for s in focus_scores)}/{len(focus_scores)}')
#     print('\nDownload gradcam_video.mp4 from the Colab Files panel (left sidebar).')


---
# Section 11 - Full Video Analysis

## What this section does

Section 9.3 processes a video and saves an annotated MP4 — but gives you almost no insight
into what happened inside it. This section fills that gap with a complete pipeline:

| Cell | What it produces |
|---|---|
| **11.0** | `VideoAnalyzer` class — full pipeline with per-frame logging |
| **11.1** | Run analysis → annotated output video |
| **11.2** | Detection timeline chart (phone confidence over every frame) |
| **11.3** | Event summary: phone-use segments, total exposure time, longest streak |
| **11.4** | Frame sampler: auto-picks representative frames (TP, FP if any, SAFE) |

## Why this matters

A raw annotated video only tells you *that* detection happened.
The analysis here tells you:
- **When** phone use occurred (start/end frame, timestamp)
- **How long** each episode lasted
- **How confident** the model was during each episode
- **What the frames actually look like** at key moments

This is what you'd show a supervisor or include in a project report.


In [ ]:
# 11.0 - VideoAnalyzer class
#
# Wraps PhoneDetector with per-frame logging so every frame's result
# (frame index, timestamp, alert, confidence, boxes) is recorded.
# The log is a list of dicts — easy to turn into a DataFrame for plotting.

import cv2, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from pathlib import Path
from tqdm import tqdm
from ultralytics import YOLO


class VideoAnalyzer:
    """
    Full video analysis pipeline for phone detection.

    Usage:
        va  = VideoAnalyzer(model_path, conf=0.25)
        log = va.process(video_path, output_path='annotated.mp4')
        df  = va.to_dataframe(log)
        va.plot_timeline(df)
        va.plot_summary_dashboard(df, video_path)
        va.sample_frames(log, video_path)
    """

    def __init__(self, model_path: str, conf: float = 0.25):
        self.model        = YOLO(model_path)
        self.conf         = conf
        self._phone_kw    = {'phone', 'cell phone', 'mobile', 'cellphone'}
        print(f'VideoAnalyzer ready  |  model={Path(model_path).name}  conf={conf}')

    # ── main pipeline ─────────────────────────────────────────────────────────

    def process(self, video_path: str, output_path: str = 'video_analyzed.mp4',
                max_frames: int = None, frame_skip: int = 1) -> list:
        """
        Process every frame of a video.

        Returns a list of per-frame dicts:
            frame_idx, timestamp_s, alert, n_phones,
            max_conf, boxes, latency_ms
        """
        cap   = cv2.VideoCapture(video_path)
        fps   = cap.get(cv2.CAP_PROP_FPS) or 30
        W     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        H     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        limit = min(max_frames or total, total)

        print(f'Video  : {Path(video_path).name}')
        print(f'Size   : {W}x{H}  |  FPS: {fps:.1f}  |  Frames: {total}  |  Processing: {limit}')
        if frame_skip > 1:
            print(f'Skip   : every {frame_skip} frames  (effective {fps/frame_skip:.1f} fps processed)')

        writer = cv2.VideoWriter(
            output_path,
            cv2.VideoWriter_fourcc(*'mp4v'),
            fps, (W, H)
        )

        log  = []
        idx  = 0

        with tqdm(total=limit, desc='Analyzing', unit='frame') as pbar:
            while cap.isOpened() and idx < limit:
                ret, frame = cap.read()
                if not ret:
                    break

                if idx % frame_skip == 0:
                    t0  = time.perf_counter()
                    res = self.model.predict(frame, conf=self.conf, verbose=False)[0]
                    lat = (time.perf_counter() - t0) * 1000

                    dets      = []
                    phone_det = []
                    if res.boxes is not None:
                        for box in res.boxes:
                            cid  = int(box.cls[0])
                            name = self.model.names.get(cid, str(cid))
                            entry = {
                                'class_name': name,
                                'confidence': float(box.conf[0]),
                                'box'       : box.xyxy[0].tolist(),
                                'is_phone'  : name.lower() in self._phone_kw,
                            }
                            dets.append(entry)
                            if entry['is_phone']:
                                phone_det.append(entry)

                    alert    = len(phone_det) > 0
                    max_conf = max((d['confidence'] for d in phone_det), default=0.0)

                    log.append({
                        'frame_idx'  : idx,
                        'timestamp_s': idx / fps,
                        'alert'      : alert,
                        'n_phones'   : len(phone_det),
                        'max_conf'   : max_conf,
                        'boxes'      : [d['box'] for d in phone_det],
                        'latency_ms' : lat,
                        'all_dets'   : dets,
                    })

                    annotated = self._draw_frame(frame.copy(), dets, idx, fps, max_conf)
                    writer.write(annotated)
                    prev_annotated = annotated

                else:
                    # Write previous frame for skipped frames
                    if log:
                        writer.write(prev_annotated if 'prev_annotated' in dir() else frame)
                    else:
                        writer.write(frame)

                idx  += 1
                pbar.update(1)

        cap.release()
        writer.release()

        phone_frames = sum(1 for r in log if r['alert'])
        print(f'\nDone. Output saved: {output_path}')
        print(f'Phone detected in {phone_frames}/{len(log)} processed frames '
              f'({phone_frames/len(log)*100:.1f}%)')
        return log

    def to_dataframe(self, log: list) -> pd.DataFrame:
        """Convert the per-frame log to a pandas DataFrame for analysis."""
        df = pd.DataFrame([{
            'frame_idx'  : r['frame_idx'],
            'timestamp_s': r['timestamp_s'],
            'alert'      : r['alert'],
            'n_phones'   : r['n_phones'],
            'max_conf'   : r['max_conf'],
            'latency_ms' : r['latency_ms'],
        } for r in log])
        return df

    # ── visualisations ────────────────────────────────────────────────────────

    def plot_timeline(self, df: pd.DataFrame, save_path: str = 'video_timeline.png'):
        """
        Two-panel chart:
          Top : phone detection confidence over time (line) with alert shading
          Bottom: binary alert bar (red=phone, green=safe) — easy to see episodes
        """
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 6),
                                        gridspec_kw={'height_ratios': [3, 1]},
                                        sharex=True)

        t   = df['timestamp_s'].values
        conf = df['max_conf'].values
        alert = df['alert'].values

        # ── confidence line ──
        ax1.plot(t, conf, color='#2c7bb6', linewidth=1.2, alpha=0.8, label='Phone confidence')
        ax1.axhline(self.conf, color='orange', linestyle='--', linewidth=1,
                    label=f'Threshold ({self.conf})')
        ax1.fill_between(t, conf, self.conf,
                         where=(conf >= self.conf),
                         alpha=0.25, color='red', label='Above threshold')
        ax1.set_ylabel('Max phone confidence', fontsize=10)
        ax1.set_ylim(0, 1.05)
        ax1.legend(fontsize=8, loc='upper right')
        ax1.set_title('Phone Detection Confidence Over Time', fontweight='bold')
        ax1.grid(True, alpha=0.3)

        # ── alert timeline bar ──
        colors = ['#e74c3c' if a else '#2ecc71' for a in alert]
        ax2.bar(t, np.ones(len(t)), width=(t[1]-t[0]) if len(t) > 1 else 1,
                color=colors, align='edge', linewidth=0)
        ax2.set_ylabel('Alert', fontsize=9)
        ax2.set_xlabel('Time (seconds)', fontsize=10)
        ax2.set_yticks([])
        red_patch   = mpatches.Patch(color='#e74c3c', label='Phone detected')
        green_patch = mpatches.Patch(color='#2ecc71', label='Safe')
        ax2.legend(handles=[red_patch, green_patch], fontsize=8, loc='upper right')
        ax2.set_title('Alert Timeline', fontweight='bold')

        plt.tight_layout()
        plt.savefig(save_path, dpi=130, bbox_inches='tight')
        plt.show()
        print(f'Saved: {save_path}')

    def plot_summary_dashboard(self, df: pd.DataFrame, video_path: str,
                               save_path: str = 'video_dashboard.png'):
        """
        4-panel summary dashboard:
          [1] Pie chart: phone vs safe frames
          [2] Confidence histogram
          [3] Detection episodes (segments above threshold)
          [4] Latency distribution
        """
        fig = plt.figure(figsize=(18, 8))
        fig.suptitle(f'Video Analysis Dashboard — {Path(video_path).name}',
                     fontsize=14, fontweight='bold')
        gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.45, wspace=0.4)

        phone_frames = int(df['alert'].sum())
        safe_frames  = len(df) - phone_frames
        fps_vid      = 1 / (df['timestamp_s'].diff().median() + 1e-9)
        duration_s   = df['timestamp_s'].max()

        # ── Panel 1: Pie ──────────────────────────────────────────────────────
        ax1 = fig.add_subplot(gs[0, 0])
        vals   = [phone_frames, safe_frames]
        labels = [f'Phone\n{phone_frames} frames', f'Safe\n{safe_frames} frames']
        clrs   = ['#e74c3c', '#2ecc71']
        wedges, texts, autotexts = ax1.pie(
            vals, labels=labels, colors=clrs, autopct='%1.1f%%',
            startangle=90, wedgeprops=dict(edgecolor='white', linewidth=1.5)
        )
        for at in autotexts:
            at.set_fontsize(9)
        ax1.set_title('Frame Classification', fontweight='bold')

        # ── Panel 2: Key stats ────────────────────────────────────────────────
        ax2 = fig.add_subplot(gs[0, 1])
        ax2.axis('off')
        phone_time_s = phone_frames / fps_vid if fps_vid else 0
        mean_conf    = df.loc[df['alert'], 'max_conf'].mean() if phone_frames else 0
        segments     = self._get_segments(df)
        longest_s    = max((s['duration_s'] for s in segments), default=0)

        stats_text = (
            f"Total duration    : {duration_s:.1f}s
"
            f"Frames processed  : {len(df)}
"
            f"Phone frames      : {phone_frames}  ({phone_frames/len(df)*100:.1f}%)
"
            f"Phone time (est.) : {phone_time_s:.1f}s
"
            f"Detection episodes: {len(segments)}
"
            f"Longest episode   : {longest_s:.1f}s
"
            f"Mean conf (phone) : {mean_conf:.3f}
"
            f"Mean latency      : {df['latency_ms'].mean():.1f}ms
"
            f"Effective FPS     : {1000/df['latency_ms'].mean():.1f}"
        )
        ax2.text(0.05, 0.95, stats_text, transform=ax2.transAxes,
                 fontsize=9.5, verticalalignment='top', fontfamily='monospace',
                 bbox=dict(boxstyle='round', facecolor='#f0f4f8', alpha=0.8))
        ax2.set_title('Key Statistics', fontweight='bold')

        # ── Panel 3: Confidence histogram ─────────────────────────────────────
        ax3 = fig.add_subplot(gs[0, 2:])
        phone_conf = df.loc[df['alert'], 'max_conf']
        if len(phone_conf):
            ax3.hist(phone_conf, bins=20, color='#e74c3c', alpha=0.75,
                     edgecolor='white', linewidth=0.5)
        ax3.axvline(self.conf, color='orange', linestyle='--', linewidth=1.5,
                    label=f'Threshold ({self.conf})')
        ax3.set_xlabel('Detection confidence', fontsize=9)
        ax3.set_ylabel('Frame count', fontsize=9)
        ax3.set_title('Confidence Distribution (phone frames only)', fontweight='bold')
        ax3.legend(fontsize=8)
        ax3.grid(True, alpha=0.3)

        # ── Panel 4: Episode table ────────────────────────────────────────────
        ax4 = fig.add_subplot(gs[1, :3])
        ax4.set_xlim(0, duration_s)
        ax4.set_ylim(0, 1)
        ax4.set_yticks([])
        ax4.set_xlabel('Time (seconds)', fontsize=9)
        ax4.set_title('Detection Episodes (each bar = one continuous phone-use event)',
                      fontweight='bold')

        # Background: full safe bar
        ax4.barh(0.5, duration_s, left=0, height=0.5, color='#2ecc71', alpha=0.3)
        for seg in segments:
            ax4.barh(0.5, seg['duration_s'], left=seg['start_s'],
                     height=0.5, color='#e74c3c', alpha=0.85)
            if seg['duration_s'] > duration_s * 0.03:   # only label wide enough bars
                ax4.text(seg['start_s'] + seg['duration_s']/2, 0.5,
                         f"{seg['duration_s']:.1f}s",
                         ha='center', va='center', fontsize=7.5,
                         color='white', fontweight='bold')

        ax4.grid(True, axis='x', alpha=0.3)

        # ── Panel 5: Latency ─────────────────────────────────────────────────
        ax5 = fig.add_subplot(gs[1, 3])
        ax5.hist(df['latency_ms'], bins=20, color='#3498db', alpha=0.75,
                 edgecolor='white', linewidth=0.5)
        ax5.axvline(df['latency_ms'].mean(), color='orange', linestyle='--',
                    linewidth=1.5, label=f"Mean {df['latency_ms'].mean():.0f}ms")
        ax5.set_xlabel('Latency (ms)', fontsize=9)
        ax5.set_ylabel('Frames', fontsize=9)
        ax5.set_title('Per-frame Latency', fontweight='bold')
        ax5.legend(fontsize=8)
        ax5.grid(True, alpha=0.3)

        plt.savefig(save_path, dpi=130, bbox_inches='tight')
        plt.show()
        print(f'Saved: {save_path}')
        return segments

    def sample_frames(self, log: list, video_path: str,
                      n_phone: int = 6, n_safe: int = 2,
                      save_path: str = 'video_sampled_frames.png'):
        """
        Pull representative frames from the video and display them in a grid.
        - Phone frames: highest-confidence detections
        - Safe frames: random selection for comparison
        """
        phone_entries = sorted(
            [r for r in log if r['alert']],
            key=lambda x: -x['max_conf']
        )[:n_phone]
        safe_entries  = [r for r in log if not r['alert']]
        safe_entries  = random.sample(safe_entries, min(n_safe, len(safe_entries)))

        all_entries = phone_entries + safe_entries
        if not all_entries:
            print('No frames to sample.')
            return

        # Seek to each frame and draw boxes
        frames_out = []
        cap = cv2.VideoCapture(video_path)
        for entry in all_entries:
            cap.set(cv2.CAP_PROP_POS_FRAMES, entry['frame_idx'])
            ret, frame = cap.read()
            if not ret:
                continue
            ann = self._draw_frame(frame, entry['all_dets'],
                                   entry['frame_idx'],
                                   cap.get(cv2.CAP_PROP_FPS) or 30,
                                   entry['max_conf'])
            frames_out.append((entry, ann))
        cap.release()

        if not frames_out:
            print('Could not read frames from video.')
            return

        cols = 4
        rows = (len(frames_out) + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols, figsize=(cols*5, rows*4))
        axes_flat = axes.flatten() if hasattr(axes, 'flatten') else [axes]

        for ax, (entry, frame_bgr) in zip(axes_flat, frames_out):
            ax.imshow(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
            ts   = entry['timestamp_s']
            mins = int(ts // 60); secs = ts % 60
            if entry['alert']:
                ax.set_title(
                    f"PHONE  conf={entry['max_conf']:.2f}\n"
                    f"t={mins:02d}:{secs:05.2f}  frame#{entry['frame_idx']}",
                    color='red', fontsize=8, fontweight='bold'
                )
            else:
                ax.set_title(
                    f"SAFE\nt={mins:02d}:{secs:05.2f}  frame#{entry['frame_idx']}",
                    color='green', fontsize=8, fontweight='bold'
                )
            ax.axis('off')
        for ax in axes_flat[len(frames_out):]:
            ax.axis('off')

        plt.suptitle('Representative Frames — Top phone detections + safe samples',
                     fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.savefig(save_path, dpi=130, bbox_inches='tight')
        plt.show()
        print(f'Saved: {save_path}')

    def print_episode_report(self, segments: list):
        """Print a readable table of every detected phone-use episode."""
        if not segments:
            print('No phone-use episodes detected.')
            return
        print()
        print('=' * 65)
        print('  PHONE-USE EPISODE REPORT')
        print('=' * 65)
        print(f'  {'#':>3}  {'Start':>8}  {'End':>8}  {'Duration':>9}  {'Frames':>7}  {'MeanConf':>9}')
        print('-' * 65)
        for i, seg in enumerate(segments, 1):
            s, e   = seg['start_s'], seg['end_s']
            sm, ss = int(s//60), s%60
            em, es = int(e//60), e%60
            print(f"  {i:>3}  {sm:02d}:{ss:05.2f}  {em:02d}:{es:05.2f}  "
                  f"{seg['duration_s']:>7.1f}s  {seg['n_frames']:>7}  {seg['mean_conf']:>9.3f}")
        print('=' * 65)
        total_s = sum(s['duration_s'] for s in segments)
        print(f'  Total phone-use time : {total_s:.1f}s across {len(segments)} episode(s)')
        print('=' * 65)

    # ── private helpers ───────────────────────────────────────────────────────

    # Classes to hide completely from video output
    _HIDDEN_CLASSES = {'wheel', 'steering wheel', 'wheel_driver'}

    def _draw_frame(self, frame, dets, frame_idx, fps, max_conf):
        """Draw detections + HUD. Wheel is hidden; driver shown as faint outline only."""
        PHONE  = (0, 0, 220)
        DRIVER = (60, 60, 60)
        has_phone = any(d.get('is_phone', False) for d in dets)

        for d in dets:
            name = d.get('class_name', '').lower()
            if name in self._HIDDEN_CLASSES:
                continue  # wheel — skip entirely
            x1,y1,x2,y2 = map(int, d['box'])
            if d.get('is_phone', False):
                cv2.rectangle(frame, (x1,y1), (x2,y2), PHONE, 3)
                lbl = f"phone {d['confidence']:.2f}"
                (tw,th),_ = cv2.getTextSize(lbl, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
                cv2.rectangle(frame, (x1,y1-th-8), (x1+tw+6,y1), PHONE, -1)
                cv2.putText(frame, lbl, (x1+3,y1-5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)
            else:
                # driver — thin faint outline, no label
                cv2.rectangle(frame, (x1,y1), (x2,y2), DRIVER, 1)

        # Banner
        banner_color = PHONE if has_phone else (0, 160, 0)
        n_ph   = sum(1 for d in dets if d['is_phone'])
        banner = f'PHONE DETECTED ({n_ph})  conf={max_conf:.2f}' if has_phone else 'SAFE'
        cv2.rectangle(frame, (0,0), (frame.shape[1], 40), banner_color, -1)
        cv2.putText(frame, banner, (10,28),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.85, (255,255,255), 2)

        # Frame counter + timestamp
        ts = frame_idx / fps
        hud = f'Frame {frame_idx:05d}  |  {int(ts//60):02d}:{ts%60:05.2f}'
        cv2.putText(frame, hud,
                    (frame.shape[1] - 280, frame.shape[0] - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200,200,200), 1)
        return frame

    def _get_segments(self, df: pd.DataFrame) -> list:
        """
        Merge consecutive alert frames into continuous episodes.
        A gap of ≤ 2 frames is bridged (handles single-frame misses in the middle).
        """
        if df.empty:
            return []

        alerts = df['alert'].values.tolist()
        ts     = df['timestamp_s'].values.tolist()
        confs  = df['max_conf'].values.tolist()
        fidxs  = df['frame_idx'].values.tolist()

        segments  = []
        in_seg    = False
        seg_start = 0
        gap_count = 0
        GAP_TOL   = 2   # bridge up to 2 non-alert frames (handles brief misses)

        seg_frames = []

        for i, alert in enumerate(alerts):
            if alert:
                if not in_seg:
                    in_seg    = True
                    seg_start = i
                    seg_frames = []
                gap_count = 0
                seg_frames.append(i)
            else:
                if in_seg:
                    gap_count += 1
                    if gap_count > GAP_TOL:
                        # Close segment
                        end_i = seg_frames[-1]
                        segments.append({
                            'start_s'  : ts[seg_start],
                            'end_s'    : ts[end_i],
                            'duration_s': ts[end_i] - ts[seg_start],
                            'n_frames' : len(seg_frames),
                            'mean_conf': float(np.mean([confs[j] for j in seg_frames])),
                            'start_frame': fidxs[seg_start],
                            'end_frame'  : fidxs[end_i],
                        })
                        in_seg    = False
                        gap_count = 0
                        seg_frames = []

        # Close any open segment at end
        if in_seg and seg_frames:
            end_i = seg_frames[-1]
            segments.append({
                'start_s'  : ts[seg_start],
                'end_s'    : ts[end_i],
                'duration_s': ts[end_i] - ts[seg_start],
                'n_frames' : len(seg_frames),
                'mean_conf': float(np.mean([confs[j] for j in seg_frames])),
                'start_frame': fidxs[seg_start],
                'end_frame'  : fidxs[end_i],
            })

        return segments


print('VideoAnalyzer class defined.')
print('Next: run cell 11.1 to analyze your video.')


In [ ]:
# 11.1 - Run full video analysis
#
# SET VIDEO_PATH to your .mp4 file before running.
# FRAME_SKIP=1 processes every frame (most accurate, slower).
# FRAME_SKIP=2 halves the time with minimal accuracy loss on typical videos.

VIDEO_PATH   = 'test_video.mp4'   # <- path to your video
OUTPUT_VIDEO = 'video_analyzed.mp4'
CONF         = 0.25
FRAME_SKIP   = 1     # 1=every frame, 2=every other frame, etc.
MAX_FRAMES   = None  # None = whole video

_bmp = get_best_model()

if not _bmp:
    print('No model found. Run Section 7 first.')
elif not Path(VIDEO_PATH).exists():
    print(f'Video not found: {VIDEO_PATH}')
    print('Upload your .mp4 to Colab (Files panel, left sidebar) and update VIDEO_PATH.')
else:
    va  = VideoAnalyzer(str(_bmp), conf=CONF)
    log = va.process(VIDEO_PATH, output_path=OUTPUT_VIDEO,
                     max_frames=MAX_FRAMES, frame_skip=FRAME_SKIP)

    # Convert to DataFrame for all subsequent cells
    df_video = va.to_dataframe(log)
    print(f'\nDataFrame shape: {df_video.shape}')
    print(df_video.head())
    print('\nRun cells 11.2 → 11.4 for charts and the episode report.')


In [ ]:
# 11.2 - Detection timeline chart
#
# Top panel : phone confidence over every frame, shaded where above threshold
# Bottom panel: simple red/green bar — instantly shows when phone was in use
#
# Good to include in your project report — visually tells the whole story.

if 'df_video' not in globals() or df_video is None:
    print('Run cell 11.1 first.')
else:
    va.plot_timeline(df_video, save_path='video_timeline.png')


In [ ]:
# 11.3 - Summary dashboard + episode report
#
# Dashboard has 5 panels:
#   [1] Pie: phone vs safe frame proportion
#   [2] Key stats text block (total time, episodes, mean conf, etc.)
#   [3] Confidence histogram for phone frames
#   [4] Episode timeline bar (each red bar = one continuous phone-use episode)
#   [5] Per-frame latency histogram
#
# Below the dashboard: a table listing every episode with start/end time,
# duration, frame count, and mean confidence.

if 'df_video' not in globals() or df_video is None:
    print('Run cell 11.1 first.')
else:
    segments = va.plot_summary_dashboard(df_video, VIDEO_PATH,
                                         save_path='video_dashboard.png')
    va.print_episode_report(segments)


In [ ]:
# 11.4 - Representative frame sampler
#
# Seeks back into the original video and pulls:
#   - N_PHONE_SAMPLES : highest-confidence phone detection frames
#   - N_SAFE_SAMPLES  : random safe frames (for visual comparison)
#
# Each frame shows the annotated detection + timestamp + frame number.
# Use this to spot-check what the model actually saw at key moments.

N_PHONE_SAMPLES = 6
N_SAFE_SAMPLES  = 2

if 'log' not in globals() or log is None:
    print('Run cell 11.1 first.')
else:
    va.sample_frames(log, VIDEO_PATH,
                     n_phone=N_PHONE_SAMPLES,
                     n_safe=N_SAFE_SAMPLES,
                     save_path='video_sampled_frames.png')


In [ ]:
# Final experiment summary
print('=' * 60)
print('  PHONE DETECTION - EXPERIMENT SUMMARY')
print('=' * 60)

cn = globals().get('CLASS_NAMES', {})
ti = globals().get('total_images', '?')
print(f'Dataset  : Cellphone Detector Drivers v2')
print(f'Classes  : {cn}')
print(f'Images   : {ti}')

tc = globals().get('TRAIN_CONFIG', {})
if tc:
    print(f'Model    : YOLOv8n (fine-tuned)')
    print(f'Epochs   : {tc.get("epochs")}')
    print(f'Img size : {tc.get("imgsz")}')
    print(f'LR       : {tc.get("lr0")} -> {tc.get("lr0",0)*tc.get("lrf",0):.5f}')

bm = get_best_model()
if bm:
    print(f'Best model: {bm}  ({bm.stat().st_size/1e6:.1f} MB)')
else:
    print('Best model: Not found - run Section 7 to train.')

print('=' * 60)
